# EquiNET polymer-hairpin: self-contained Colab notebook

This notebook contains the trajectory simulation, work-distribution simulation, full-coordinate EquiNET inference, coarse-grained inference, and the final cumulative-estimate figure.

**Requested production settings are retained:** `epsilon_native = 3.0`, 1000 trajectories, and a disjoint 500/500 training/test split. The former 20,000-step fixed waiting stage is absorbed into the initial equilibration, so `n_steps_eq_xy_start = 170000`.

### Google Colab
1. In Colab choose **Runtime > Change runtime type > T4 GPU** (or another GPU) for the EquiNET training cells.
2. Run the setup cell below first.
3. Run the notebook from top to bottom. Simulation is CPU-bound; the EGNN inference automatically uses CUDA when a Colab GPU is available.
4. Outputs are written under `/content/equinet_hairpin_notebook_run` on Colab. Enable Google Drive in the setup cell if you want outputs to survive runtime resets.

> **Compute note:** the requested 1000-trajectory simulations are production-scale CPU calculations. The notebook is Colab-compatible, but a free Colab session may not remain connected long enough to finish the full trajectory and especially the long work-distribution calculation. The cluster workflow remains preferable for the complete production run.


## 0. Colab setup


In [1]:
# Colab already includes NumPy, Matplotlib and PyTorch. This installs tqdm if needed.
%pip install -q tqdm

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
USE_GOOGLE_DRIVE = False  # Set True before running this cell to persist outputs in Drive.

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_ROOT = Path('/content/drive/MyDrive/EquiNET_colab')
elif IN_COLAB:
    COLAB_ROOT = Path('/content')
else:
    COLAB_ROOT = Path.cwd()

print('Running in Colab:', IN_COLAB)
print('Base directory:', COLAB_ROOT)


Running in Colab: True
Base directory: /content


## 0. Notebook paths and requested dataset sizes


In [2]:
from pathlib import Path
import multiprocessing as mp

ROOT = (COLAB_ROOT / "equinet_hairpin_notebook_run").resolve()
TRAJ_DIR = ROOT / "trajectory_data"
WORK_DIR = ROOT / "work_distribution"
FULL_INFERENCE_DIR = ROOT / "inference_full"
CG_INFERENCE_DIR = ROOT / "inference_cg"

for directory in [TRAJ_DIR, WORK_DIR, FULL_INFERENCE_DIR, CG_INFERENCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Requested dataset split
N_TOTAL = 100
N_TRAIN = 50
N_TEST = 50
INFERENCE_SEED = 0

# Same trajectory-simulation defaults as the supplied script.
NOTEBOOK_N_WORKERS = max(1, mp.cpu_count() - 1)
NOTEBOOK_CHUNK_SIZE = 100
NOTEBOOK_BASE_SEED = 123456

# `spawn` cannot directly import worker functions defined inside a notebook.
# On Linux/macOS notebooks we therefore use `fork`; Windows has no `fork`.
NOTEBOOK_MP_CONTEXT = "fork" if hasattr(mp, "get_context") and __import__("sys").platform != "win32" else "spawn"

print(f"Output root: {ROOT}")
print(f"Trajectories: {N_TOTAL} total = {N_TRAIN} train + {N_TEST} test")


Output root: /content/equinet_hairpin_notebook_run
Trajectories: 100 total = 50 train + 50 test


In [3]:
import logging

logging.getLogger("fontTools").setLevel(logging.ERROR)

## 1. Trajectory simulation

Generates `traj_fwd.npy`, `traj_rev.npy`, `trap_fwd.npy`, and `trap_rev.npy` in `TRAJ_DIR`. The underlying dynamics use the supplied parameters, with the requested 170,000-step initial equilibration and no separate 20,000-step fixed stage. Configurations are stored every 5000 simulation steps. Running this cell starts the full 1000-trajectory simulation.


In [ ]:
import os
import sys
import math
import multiprocessing as mp

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np

# ============================================================
# 0) Notebook configuration
# ============================================================
OUTPUT_DIR = str(TRAJ_DIR)
N_total = N_TOTAL
N_WORKERS = NOTEBOOK_N_WORKERS
CHUNK_SIZE = NOTEBOOK_CHUNK_SIZE
BASE_SEED = NOTEBOOK_BASE_SEED

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# Parameters
# ================================================================
dt = 1e-4 #use 1e-5 for production

n_steps_eq_xy_start = 17000
n_steps_x_drive = 60000
n_steps_eq_end = 15000

gamma = 1.0
T = 1.0
kB = 1.0

n_beads = 13
dim = 3

k_bond = 500.0
k_bend = 20.0

k_anchor = 1000.0
k_pull = 10.0

v_x = 2.0

epsilon_native = 3.0 #was 3
epsilon_rep = 1.0
sigma_rep = 0.8

store_stride = 500

sqrt_2Tdt_over_gamma = np.sqrt(2.0 * kB * T * dt / gamma)

# Fixed trap start/end coordinate
TRAP_START = np.array([1.3548766, -1.8808337, -0.00921113], dtype=np.float64)

# ================================================================
# Native contacts
# ================================================================
native_pairs_1based = [(4, 7), (3, 8), (2, 9)]
native_pairs = [(i - 1, j - 1) for i, j in native_pairs_1based]
native_set = {tuple(sorted(p)) for p in native_pairs}

# ================================================================
# Initial folded hairpin guess
# ================================================================
def make_initial_hairpin_guess():
    pts2 = np.array([
        [0.0, 0.0],
        [0.0, 1.0],
        [0.0, 2.0],
        [0.0, 3.0],
        [0.0, 4.0],
        [0.4, 4.8],
        [1.2, 4.2],
        [1.2, 3.2],
        [1.2, 2.2],
        [1.2, 1.2],
        [1.2, 0.2],
        [1.2, -0.8],
        [1.2, -1.8],
    ], dtype=np.float64)

    pts3 = np.zeros((n_beads, 3), dtype=np.float64)
    pts3[:, :2] = pts2
    return pts3

# ================================================================
# LJ helpers
# ================================================================
def U_LJ_scalar(r, epsilon, sigma):
    r = np.maximum(r, 1e-12)
    inv_r6 = (sigma / r) ** 6
    return 4.0 * epsilon * (inv_r6**2 - inv_r6)

def F_LJ_vec(dr, epsilon, sigma):
    r2 = np.sum(dr * dr, axis=-1, keepdims=True)
    r2 = np.maximum(r2, 1e-24)
    r = np.sqrt(r2)
    pref = 24.0 * epsilon * (
        2.0 * sigma**12 / r**14 - sigma**6 / r**8
    )
    return pref * dr

# ================================================================
# Single-configuration mechanics
# ================================================================
def F_bonds_single(r, bond_rest):
    F = np.zeros_like(r)
    dr = r[1:] - r[:-1]
    dist = np.linalg.norm(dr, axis=1, keepdims=True)
    dist = np.maximum(dist, 1e-12)
    rest = bond_rest[:, None]
    fpair = k_bond * (dist - rest) * dr / dist
    F[:-1] += fpair
    F[1:] -= fpair
    return F

def F_bend_single(r, b_ref):
    F = np.zeros_like(r)
    a = k_bend / b_ref**2
    c = r[2:] - 2.0 * r[1:-1] + r[:-2]

    for i in range(len(c)):
        ci = c[i]
        F[i] += -a * ci
        F[i + 1] += 2.0 * a * ci
        F[i + 2] += -a * ci

    return F

def F_native_contacts_single(r, native_sigma):
    F = np.zeros_like(r)

    for pair_idx, (i, j) in enumerate(native_pairs):
        dr = r[i] - r[j]
        fij = F_LJ_vec(dr[None, :], epsilon_native, native_sigma[pair_idx])[0]
        F[i] += fij
        F[j] -= fij

    return F

def F_repulsive_single(r):
    F = np.zeros_like(r)
    rcut = 2.0 ** (1.0 / 6.0) * sigma_rep

    for i in range(n_beads):
        for j in range(i + 1, n_beads):
            if abs(i - j) == 1:
                continue
            if (i, j) in native_set:
                continue

            dr = r[i] - r[j]
            rij = np.linalg.norm(dr)

            if rij < rcut:
                fij = F_LJ_vec(dr[None, :], epsilon_rep, sigma_rep)[0]
                F[i] += fij
                F[j] -= fij

    return F

def F_anchor_single(r1, r1_trap):
    return -k_anchor * (r1 - r1_trap)

def F_pull_single(r_end, trap_end):
    return -k_pull * (r_end - trap_end)

def total_conservative_forces_single(
    r, bond_rest, b_ref, native_sigma, r1_trap, trap_end
):
    F = np.zeros_like(r)
    F += F_bonds_single(r, bond_rest)
    F += F_bend_single(r, b_ref)
    F += F_native_contacts_single(r, native_sigma)
    F += F_repulsive_single(r)
    F[0] += F_anchor_single(r[0], r1_trap)
    F[-1] += F_pull_single(r[-1], trap_end)
    return F

# ================================================================
# Batched mechanics
# ================================================================
def F_bonds_batch(R, bond_rest):
    F = np.zeros_like(R)
    dr = R[:, 1:] - R[:, :-1]
    dist = np.linalg.norm(dr, axis=2, keepdims=True)
    dist = np.maximum(dist, 1e-12)
    rest = bond_rest[None, :, None]
    fpair = k_bond * (dist - rest) * dr / dist
    F[:, :-1] += fpair
    F[:, 1:] -= fpair
    return F

def F_bend_batch(R, b_ref):
    F = np.zeros_like(R)
    a = k_bend / b_ref**2
    c = R[:, 2:] - 2.0 * R[:, 1:-1] + R[:, :-2]
    F[:, :-2] += -a * c
    F[:, 1:-1] += 2.0 * a * c
    F[:, 2:] += -a * c
    return F

def F_native_contacts_batch(R, native_sigma):
    F = np.zeros_like(R)

    for pair_idx, (i, j) in enumerate(native_pairs):
        dr = R[:, i] - R[:, j]
        fij = F_LJ_vec(dr, epsilon_native, native_sigma[pair_idx])
        F[:, i] += fij
        F[:, j] -= fij

    return F

def F_repulsive_batch(R):
    F = np.zeros_like(R)
    rcut = 2.0 ** (1.0 / 6.0) * sigma_rep

    for i in range(n_beads):
        for j in range(i + 1, n_beads):
            if abs(i - j) == 1:
                continue
            if (i, j) in native_set:
                continue

            dr = R[:, i] - R[:, j]
            rij = np.linalg.norm(dr, axis=1)
            mask = rij < rcut

            if np.any(mask):
                fij = F_LJ_vec(dr[mask], epsilon_rep, sigma_rep)
                F[mask, i] += fij
                F[mask, j] -= fij

    return F

def F_anchor_batch(r1, r1_trap):
    return -k_anchor * (r1 - r1_trap[None, :])

def F_pull_batch(r_end, trap_end):
    return -k_pull * (r_end - trap_end[None, :])

def total_conservative_forces_batch(
    R, bond_rest, b_ref, native_sigma, r1_trap, trap_end
):
    F = np.zeros_like(R)
    F += F_bonds_batch(R, bond_rest)
    F += F_bend_batch(R, b_ref)
    F += F_native_contacts_batch(R, native_sigma)
    F += F_repulsive_batch(R)
    F[:, 0] += F_anchor_batch(R[:, 0], r1_trap)
    F[:, -1] += F_pull_batch(R[:, -1], trap_end)
    return F

# ================================================================
# Mechanical relaxation
# ================================================================
def relax_guess_to_folded_shape(
    r_guess, max_steps=200000, dt_relax=2e-4, tol=1e-8
):
    r = r_guess.copy()

    bond_rest_tmp = np.linalg.norm(r_guess[1:] - r_guess[:-1], axis=1)
    b_ref_tmp = np.mean(bond_rest_tmp)

    native_r0_tmp = np.array([
        np.linalg.norm(r_guess[i] - r_guess[j]) for i, j in native_pairs
    ], dtype=np.float64)
    native_sigma_tmp = native_r0_tmp / (2.0 ** (1.0 / 6.0))

    r1_trap = r_guess[0].copy()
    trap_end = r_guess[-1].copy()

    for step in range(max_steps):
        F = total_conservative_forces_single(
            r,
            bond_rest_tmp,
            b_ref_tmp,
            native_sigma_tmp,
            r1_trap,
            trap_end,
        )

        max_force = np.max(np.linalg.norm(F, axis=1))

        if max_force < tol:
            print(f"Mechanical pre-relaxation converged in {step} steps.")
            return r

        r += (F / gamma) * dt_relax

        if step % 20000 == 0:
            print(f"Pre-relax step {step}, max|F| = {max_force:.3e}")

    print("Warning: pre-relaxation hit max_steps.")
    return r

def build_equilibrium_model_from_relaxed_shape(r_relaxed):
    bond_rest = np.linalg.norm(r_relaxed[1:] - r_relaxed[:-1], axis=1)
    b_ref = np.mean(bond_rest)

    native_r0 = np.array([
        np.linalg.norm(r_relaxed[i] - r_relaxed[j]) for i, j in native_pairs
    ], dtype=np.float64)
    native_sigma = native_r0 / (2.0 ** (1.0 / 6.0))

    r1_trap = r_relaxed[0].copy()
    end_trap_0 = r_relaxed[-1].copy()

    return bond_rest, b_ref, native_sigma, r1_trap, end_trap_0

# ================================================================
# Protocol: fixed y/z, x-only pulling
# ================================================================
def build_trap_protocol():
    x0, y0, z0 = TRAP_START

    x_stage1 = np.full(n_steps_eq_xy_start, x0)
    y_stage1 = np.full(n_steps_eq_xy_start, y0)
    z_stage1 = np.full(n_steps_eq_xy_start, z0)

    # x-only pulling
    x_stage3 = x0 + np.linspace(
        0.0,
        v_x * dt * (n_steps_x_drive - 1),
        n_steps_x_drive,
    )
    y_stage3 = np.full(n_steps_x_drive, y0)
    z_stage3 = np.full(n_steps_x_drive, z0)

    x_stage4 = np.full(n_steps_eq_end, x_stage3[-1])
    y_stage4 = np.full(n_steps_eq_end, y0)
    z_stage4 = np.full(n_steps_eq_end, z0)

    x_protocol = np.concatenate([x_stage1, x_stage3, x_stage4])
    y_protocol = np.concatenate([y_stage1, y_stage3, y_stage4])
    z_protocol = np.concatenate([z_stage1, z_stage3, z_stage4])

    trap_fwd = np.column_stack(
        [x_protocol, y_protocol, z_protocol]
    ).astype(np.float64)

    trap_rev = trap_fwd[::-1].copy()

    return trap_fwd, trap_rev

# ================================================================
# Batched thermalization
# ================================================================
def thermalize_batch(
    R, bond_rest, b_ref, native_sigma, r1_trap, trap_end_fixed, n_steps_therm, rng
):
    for _ in range(n_steps_therm):
        F = total_conservative_forces_batch(
            R, bond_rest, b_ref, native_sigma, r1_trap, trap_end_fixed
        )
        noise = sqrt_2Tdt_over_gamma * rng.standard_normal(R.shape)
        R += (F / gamma) * dt + noise

    return R

# ================================================================
# Batched protocol simulation with storage
# ================================================================
def simulate_protocol_store_batch(
    R, trap_protocol, bond_rest, b_ref, native_sigma, r1_trap, rng
):
    M = R.shape[0]
    n_steps_total = len(trap_protocol)
    n_store = (n_steps_total - 1) // store_stride + 1

    traj = np.empty((M, n_store, n_beads, dim), dtype=np.float32)
    traj[:, 0] = R.astype(np.float32)

    store_idx = 1

    for t in range(1, n_steps_total):
        trap_old = trap_protocol[t - 1]

        F = total_conservative_forces_batch(
            R, bond_rest, b_ref, native_sigma, r1_trap, trap_old
        )
        noise = sqrt_2Tdt_over_gamma * rng.standard_normal(R.shape)
        R += (F / gamma) * dt + noise

        if t % store_stride == 0:
            traj[:, store_idx] = R.astype(np.float32)
            store_idx += 1

    return traj, R

# ================================================================
# Globals for workers
# ================================================================
G = {}

def init_worker(params):
    global G
    G = params

def worker_simulate_chunk(task):
    start, stop, seed = task
    M = stop - start

    rng = np.random.default_rng(seed)

    bond_rest = G["bond_rest"]
    b_ref = G["b_ref"]
    native_sigma = G["native_sigma"]
    r1_trap = G["r1_trap"]
    r_eq = G["r_eq"]
    trap_fwd = G["trap_fwd"]
    trap_rev = G["trap_rev"]
    traj_fwd_path = G["traj_fwd_path"]
    traj_rev_path = G["traj_rev_path"]
    shape = G["traj_shape"]

    R0 = np.tile(r_eq[None, :, :], (M, 1, 1)).astype(np.float64)

    R0 = thermalize_batch(
        R0,
        bond_rest,
        b_ref,
        native_sigma,
        r1_trap,
        trap_fwd[0],
        n_steps_eq_xy_start,
        rng,
    )

    traj_fwd_batch, Rf = simulate_protocol_store_batch(
        R0.copy(),
        trap_fwd,
        bond_rest,
        b_ref,
        native_sigma,
        r1_trap,
        rng,
    )

    traj_rev_batch, _ = simulate_protocol_store_batch(
        Rf.copy(),
        trap_rev,
        bond_rest,
        b_ref,
        native_sigma,
        r1_trap,
        rng,
    )

    mm_fwd = np.lib.format.open_memmap(
        traj_fwd_path, mode="r+", dtype=np.float32, shape=shape
    )
    mm_rev = np.lib.format.open_memmap(
        traj_rev_path, mode="r+", dtype=np.float32, shape=shape
    )

    mm_fwd[start:stop] = traj_fwd_batch
    mm_rev[start:stop] = traj_rev_batch

    mm_fwd.flush()
    mm_rev.flush()

    del mm_fwd, mm_rev, traj_fwd_batch, traj_rev_batch, R0, Rf

    print(f"Finished chunk {start}:{stop}")
    return start, stop

# ================================================================
# Main
# ================================================================
def main():
    print("Building initial 3D hairpin guess...")
    r_guess = make_initial_hairpin_guess()

    print("Relaxing rough guess to folded mechanical state...")
    r_relaxed = relax_guess_to_folded_shape(r_guess)

    print("Freezing equilibrium geometry into model parameters...")
    bond_rest, b_ref, native_sigma, r1_trap, end_trap_0 = (
        build_equilibrium_model_from_relaxed_shape(r_relaxed)
    )

    r_eq = r_relaxed.copy()

    print("Building fixed-y, x-only trap protocol...")
    trap_fwd, trap_rev = build_trap_protocol()

    print("Forward trap starts at:", trap_fwd[0])
    print("Forward trap ends at:  ", trap_fwd[-1])
    print("Reverse trap starts at:", trap_rev[0])
    print("Reverse trap ends at:  ", trap_rev[-1])

    n_steps_total = len(trap_fwd)
    n_store = (n_steps_total - 1) // store_stride + 1

    traj_shape = (N_total, n_store, n_beads, dim)

    bytes_per_file = np.prod(traj_shape) * np.dtype(np.float32).itemsize

    print(f"N_total         = {N_total}")
    print(f"n_steps_total   = {n_steps_total}")
    print(f"store_stride    = {store_stride}")
    print(f"n_store         = {n_store}")
    print(f"traj shape      = {traj_shape}")
    print(f"Each file size  ≈ {bytes_per_file / 1e9:.2f} GB")
    print(f"Total output    ≈ {2 * bytes_per_file / 1e9:.2f} GB")

    traj_fwd_path = os.path.join(OUTPUT_DIR, "traj_fwd.npy")
    traj_rev_path = os.path.join(OUTPUT_DIR, "traj_rev.npy")

    print("Creating output memmaps...")
    np.lib.format.open_memmap(
        traj_fwd_path, mode="w+", dtype=np.float32, shape=traj_shape
    )
    np.lib.format.open_memmap(
        traj_rev_path, mode="w+", dtype=np.float32, shape=traj_shape
    )

    # Optional: save protocols too
    np.save(os.path.join(OUTPUT_DIR, "trap_fwd.npy"), trap_fwd)
    np.save(os.path.join(OUTPUT_DIR, "trap_rev.npy"), trap_rev)

    params = {
        "bond_rest": bond_rest,
        "b_ref": b_ref,
        "native_sigma": native_sigma,
        "r1_trap": r1_trap,
        "r_eq": r_eq,
        "trap_fwd": trap_fwd,
        "trap_rev": trap_rev,
        "traj_fwd_path": traj_fwd_path,
        "traj_rev_path": traj_rev_path,
        "traj_shape": traj_shape,
    }

    tasks = []
    seed_seq = np.random.SeedSequence(BASE_SEED)
    child_seeds = seed_seq.spawn(math.ceil(N_total / CHUNK_SIZE))

    chunk_id = 0
    for start in range(0, N_total, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, N_total)
        seed = int(child_seeds[chunk_id].generate_state(1)[0])
        tasks.append((start, stop, seed))
        chunk_id += 1

    print(f"Running with {N_WORKERS} workers, chunk size {CHUNK_SIZE}...")

    ctx = mp.get_context(NOTEBOOK_MP_CONTEXT)
    with ctx.Pool(
        processes=N_WORKERS,
        initializer=init_worker,
        initargs=(params,),
    ) as pool:
        for _ in pool.imap_unordered(worker_simulate_chunk, tasks):
            pass

    print("\nDone. Saved:")
    print("  traj_fwd.npy")
    print("  traj_rev.npy")
    print("  trap_fwd.npy")
    print("  trap_rev.npy")

main()


Building initial 3D hairpin guess...
Relaxing rough guess to folded mechanical state...
Pre-relax step 0, max|F| = 5.101e+01
Pre-relax step 20000, max|F| = 1.131e-01
Pre-relax step 40000, max|F| = 6.762e-02
Pre-relax step 60000, max|F| = 4.194e-02
Pre-relax step 80000, max|F| = 2.659e-02
Pre-relax step 100000, max|F| = 1.710e-02
Pre-relax step 120000, max|F| = 1.109e-02
Pre-relax step 140000, max|F| = 7.231e-03
Pre-relax step 160000, max|F| = 4.733e-03
Pre-relax step 180000, max|F| = 3.105e-03
Freezing equilibrium geometry into model parameters...
Building fixed-y, x-only trap protocol...
Forward trap starts at: [ 1.3548766  -1.8808337  -0.00921113]
Forward trap ends at:   [ 1.33546766e+01 -1.88083370e+00 -9.21113000e-03]
Reverse trap starts at: [ 1.33546766e+01 -1.88083370e+00 -9.21113000e-03]
Reverse trap ends at:   [ 1.3548766  -1.8808337  -0.00921113]
N_total         = 100
n_steps_total   = 92000
store_stride    = 500
n_store         = 184
traj shape      = (100, 184, 13, 3)
Each f

## 2. Work-distribution simulation

Runs the supplied work-distribution calculation directly in the notebook for 1000 realizations. The requested 170,000-step initial equilibration/no-separate-fixed-stage change is applied here as well. All other parameters from the supplied work code are retained, including `store_stride = 500`, `n_steps_x_drive = 6000000`, and `v_x = 0.2`.


In [ ]:
"""
Batch-oriented 3D hairpin pulling work simulation.

Designed for Slurm array execution. Each array task runs an independent batch
of realizations with a task-specific random seed and writes a compact set of
outputs that can be merged later.

Environment variables:
    N_REALIZATIONS       Number of realizations in this task (default: 1000)
    BASE_SEED            Base RNG seed (default: 123456)
    SLURM_ARRAY_TASK_ID  Added to BASE_SEED when available

Run:
    python hairpin_work_array.py <OUTPUT_DIR>
"""

import os
import sys
import time
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np


# ============================================================
# Notebook task configuration
# ============================================================
OUTPUT_DIR = Path(WORK_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N = N_TOTAL
TASK_ID = 0
BASE_SEED = NOTEBOOK_BASE_SEED
SEED = BASE_SEED + TASK_ID

if N <= 0:
    raise ValueError("N_REALIZATIONS must be positive")

rng = np.random.default_rng(SEED)

print(f"Output directory: {OUTPUT_DIR}", flush=True)
print(f"Exists: {OUTPUT_DIR.exists()}", flush=True)

# Test write
test_file = OUTPUT_DIR / "write_test.txt"

with open(test_file, "w") as f:
    f.write("Hello from hairpin_work_array.py\n")
    f.write(f"TASK_ID = {TASK_ID}\n")
    f.write(f"SEED = {SEED}\n")

print(f"Wrote test file: {test_file}", flush=True)

# Also test NumPy writing
np.save(OUTPUT_DIR / "write_test.npy", np.array([1, 2, 3]))
print("NumPy write succeeded.", flush=True)
# ================================================================
# Parameters
# ================================================================
dt = 1e-4
store_stride = 50
dt_store = dt * store_stride

n_steps_eq_xy_start = 17000
n_steps_x_drive = 60000
n_steps_eq_end = 15000

gamma = 1.0
T = 1.0
kB = 1.0

n_beads = 13
dim = 3

k_bond = 500.0
k_bend = 20.0

k_anchor = 1000.0
k_pull = 10.0

v_x = 2.0 #was 2.0

epsilon_native = 3.0
epsilon_rep = 1.0
sigma_rep = 0.8

sqrt_2Tdt_over_gamma = np.sqrt(2.0 * kB * T * dt / gamma)

TRAP_START = np.array(
    [1.3548766, -1.8808337, -0.00921113], dtype=np.float64
)

PROGRESS_EVERY = int(os.environ.get("PROGRESS_EVERY", "5000"))


# ================================================================
# Native and repulsive contacts
# ================================================================
native_pairs_1based = [(4, 7), (3, 8), (2, 9)]
native_pairs = [(i - 1, j - 1) for i, j in native_pairs_1based]
native_set = {tuple(sorted(pair)) for pair in native_pairs}

repulsive_pairs = [
    (i, j)
    for i in range(n_beads)
    for j in range(i + 1, n_beads)
    if abs(i - j) != 1 and (i, j) not in native_set
]


# ================================================================
# Initial hairpin
# ================================================================
def make_initial_hairpin_guess():
    pts2 = np.array(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [0.0, 2.0],
            [0.0, 3.0],
            [0.0, 4.0],
            [0.4, 4.8],
            [1.2, 4.2],
            [1.2, 3.2],
            [1.2, 2.2],
            [1.2, 1.2],
            [1.2, 0.2],
            [1.2, -0.8],
            [1.2, -1.8],
        ],
        dtype=np.float64,
    )

    pts3 = np.zeros((n_beads, dim), dtype=np.float64)
    pts3[:, :2] = pts2
    return pts3


# ================================================================
# Forces
# ================================================================
def F_LJ_vec(dr, epsilon, sigma):
    r2 = np.sum(dr * dr, axis=-1, keepdims=True)
    r2 = np.maximum(r2, 1e-24)

    inv_r2 = 1.0 / r2
    inv_r6 = inv_r2**3
    inv_r8 = inv_r6 * inv_r2
    inv_r14 = inv_r8 * inv_r6

    pref = 24.0 * epsilon * (
        2.0 * sigma**12 * inv_r14 - sigma**6 * inv_r8
    )
    return pref * dr


def F_bonds_single(r, bond_rest):
    F = np.zeros_like(r)
    dr = r[1:] - r[:-1]
    dist = np.linalg.norm(dr, axis=1, keepdims=True)
    dist = np.maximum(dist, 1e-12)
    fpair = k_bond * (dist - bond_rest[:, None]) * dr / dist
    F[:-1] += fpair
    F[1:] -= fpair
    return F


def F_bend_single(r, b_ref):
    F = np.zeros_like(r)
    a = k_bend / b_ref**2
    c = r[2:] - 2.0 * r[1:-1] + r[:-2]
    F[:-2] += -a * c
    F[1:-1] += 2.0 * a * c
    F[2:] += -a * c
    return F


def F_native_contacts_single(r, native_sigma):
    F = np.zeros_like(r)
    for p, (i, j) in enumerate(native_pairs):
        dr = r[i] - r[j]
        fij = F_LJ_vec(dr[None, :], epsilon_native, native_sigma[p])[0]
        F[i] += fij
        F[j] -= fij
    return F


def F_repulsive_single(r):
    F = np.zeros_like(r)
    rcut2 = (2.0 ** (1.0 / 6.0) * sigma_rep) ** 2

    for i, j in repulsive_pairs:
        dr = r[i] - r[j]
        r2 = float(np.dot(dr, dr))
        if r2 < rcut2:
            fij = F_LJ_vec(dr[None, :], epsilon_rep, sigma_rep)[0]
            F[i] += fij
            F[j] -= fij
    return F


def total_forces_single(r, bond_rest, b_ref, native_sigma, r1_trap, trap_end):
    F = F_bonds_single(r, bond_rest)
    F += F_bend_single(r, b_ref)
    F += F_native_contacts_single(r, native_sigma)
    F += F_repulsive_single(r)
    F[0] += -k_anchor * (r[0] - r1_trap)
    F[-1] += -k_pull * (r[-1] - trap_end)
    return F


def F_bonds_batch(R, bond_rest):
    F = np.zeros_like(R)
    dr = R[:, 1:] - R[:, :-1]
    dist = np.linalg.norm(dr, axis=2, keepdims=True)
    dist = np.maximum(dist, 1e-12)
    fpair = k_bond * (dist - bond_rest[None, :, None]) * dr / dist
    F[:, :-1] += fpair
    F[:, 1:] -= fpair
    return F


def F_bend_batch(R, b_ref):
    F = np.zeros_like(R)
    a = k_bend / b_ref**2
    c = R[:, 2:] - 2.0 * R[:, 1:-1] + R[:, :-2]
    F[:, :-2] += -a * c
    F[:, 1:-1] += 2.0 * a * c
    F[:, 2:] += -a * c
    return F


def F_native_contacts_batch(R, native_sigma):
    F = np.zeros_like(R)
    for p, (i, j) in enumerate(native_pairs):
        dr = R[:, i] - R[:, j]
        fij = F_LJ_vec(dr, epsilon_native, native_sigma[p])
        F[:, i] += fij
        F[:, j] -= fij
    return F


def F_repulsive_batch(R):
    F = np.zeros_like(R)
    rcut2 = (2.0 ** (1.0 / 6.0) * sigma_rep) ** 2

    for i, j in repulsive_pairs:
        dr = R[:, i] - R[:, j]
        r2 = np.einsum("ij,ij->i", dr, dr)
        mask = r2 < rcut2

        if np.any(mask):
            fij = F_LJ_vec(dr[mask], epsilon_rep, sigma_rep)
            F[mask, i] += fij
            F[mask, j] -= fij

    return F


def total_forces_batch(R, bond_rest, b_ref, native_sigma, r1_trap, trap_end):
    F = F_bonds_batch(R, bond_rest)
    F += F_bend_batch(R, b_ref)
    F += F_native_contacts_batch(R, native_sigma)
    F += F_repulsive_batch(R)
    F[:, 0] += -k_anchor * (R[:, 0] - r1_trap[None, :])
    F[:, -1] += -k_pull * (R[:, -1] - trap_end[None, :])
    return F


# ================================================================
# Equilibrium model
# ================================================================
def relax_guess_to_folded_shape(
    r_guess,
    max_steps=200_000,
    dt_relax=2e-4,
    tol=1e-8,
):
    r = r_guess.copy()
    bond_rest_tmp = np.linalg.norm(r_guess[1:] - r_guess[:-1], axis=1)
    b_ref_tmp = np.mean(bond_rest_tmp)
    native_r0_tmp = np.array(
        [np.linalg.norm(r_guess[i] - r_guess[j]) for i, j in native_pairs],
        dtype=np.float64,
    )
    native_sigma_tmp = native_r0_tmp / (2.0 ** (1.0 / 6.0))
    r1_trap = r_guess[0].copy()
    trap_end = r_guess[-1].copy()

    for step in range(max_steps):
        F = total_forces_single(
            r,
            bond_rest_tmp,
            b_ref_tmp,
            native_sigma_tmp,
            r1_trap,
            trap_end,
        )
        max_force = np.max(np.linalg.norm(F, axis=1))

        if max_force < tol:
            print(f"Mechanical pre-relaxation converged in {step} steps.", flush=True)
            return r

        r += (F / gamma) * dt_relax

        if step % 20_000 == 0:
            print(
                f"Pre-relax step {step}, max|F| = {max_force:.3e}",
                flush=True,
            )

    print("Warning: pre-relaxation hit max_steps.", flush=True)
    return r


def build_equilibrium_model_from_relaxed_shape(r_relaxed):
    bond_rest = np.linalg.norm(r_relaxed[1:] - r_relaxed[:-1], axis=1)
    b_ref = np.mean(bond_rest)
    native_r0 = np.array(
        [np.linalg.norm(r_relaxed[i] - r_relaxed[j]) for i, j in native_pairs],
        dtype=np.float64,
    )
    native_sigma = native_r0 / (2.0 ** (1.0 / 6.0))
    r1_trap = r_relaxed[0].copy()
    end_trap_0 = r_relaxed[-1].copy()
    return bond_rest, b_ref, native_sigma, r1_trap, end_trap_0


# ================================================================
# Trap protocol
# ================================================================
def build_trap_protocol():
    x0, y0, z0 = TRAP_START

    x_stage1 = np.full(n_steps_eq_xy_start, x0)
    y_stage1 = np.full(n_steps_eq_xy_start, y0)
    z_stage1 = np.full(n_steps_eq_xy_start, z0)

    x_stage3 = x0 + np.linspace(
        0.0,
        v_x * dt * (n_steps_x_drive - 1),
        n_steps_x_drive,
    )
    y_stage3 = np.full(n_steps_x_drive, y0)
    z_stage3 = np.full(n_steps_x_drive, z0)

    x_stage4 = np.full(n_steps_eq_end, x_stage3[-1])
    y_stage4 = np.full(n_steps_eq_end, y0)
    z_stage4 = np.full(n_steps_eq_end, z0)

    trap_fwd = np.column_stack(
        [
            np.concatenate([x_stage1, x_stage3, x_stage4]),
            np.concatenate([y_stage1, y_stage3, y_stage4]),
            np.concatenate([z_stage1, z_stage3, z_stage4]),
        ]
    ).astype(np.float64)

    return trap_fwd, trap_fwd[::-1].copy()


# ================================================================
# Progress reporting
# ================================================================
def report_progress(label, step, total_steps, start_time):
    if step <= 0:
        return
    elapsed = time.perf_counter() - start_time
    rate = step / elapsed if elapsed > 0 else 0.0
    remaining = (total_steps - step) / rate if rate > 0 else float("inf")
    print(
        f"{label}: {step}/{total_steps} "
        f"({100.0 * step / total_steps:.1f}%), "
        f"{rate:.2f} steps/s, ETA {remaining / 3600.0:.2f} h",
        flush=True,
    )


# ================================================================
# Simulation
# ================================================================
def thermalize_batch(
    R,
    bond_rest,
    b_ref,
    native_sigma,
    r1_trap,
    trap_end_fixed,
    rng,
):
    start = time.perf_counter()

    for step in range(1, n_steps_eq_xy_start + 1):
        F = total_forces_batch(
            R,
            bond_rest,
            b_ref,
            native_sigma,
            r1_trap,
            trap_end_fixed,
        )
        R += (F / gamma) * dt
        R += sqrt_2Tdt_over_gamma * rng.standard_normal(R.shape)

        if PROGRESS_EVERY > 0 and step % PROGRESS_EVERY == 0:
            report_progress("Thermalization", step, n_steps_eq_xy_start, start)

    return R


def simulate_protocol_work_batch(
    R,
    trap_protocol,
    bond_rest,
    b_ref,
    native_sigma,
    r1_trap,
    rng,
    label,
):
    M = R.shape[0]
    n_steps_total = len(trap_protocol)
    n_store = (n_steps_total - 1) // store_stride + 1

    # Store only ensemble means plus final per-realization work.
    mean_power_t = np.empty(n_store, dtype=np.float64)
    mean_work_t = np.empty(n_store, dtype=np.float64)
    trap_store = np.empty((n_store, dim), dtype=np.float64)

    mean_power_t[0] = 0.0
    mean_work_t[0] = 0.0
    trap_store[0] = trap_protocol[0]

    cumulative_work = np.zeros(M, dtype=np.float64)
    store_idx = 1
    start = time.perf_counter()

    for t in range(1, n_steps_total):
        trap_old = trap_protocol[t - 1]
        trap_new = trap_protocol[t]

        # Only the old end-bead position is needed for midpoint work.
        r_end_old = R[:, -1, :].copy()

        F = total_forces_batch(
            R,
            bond_rest,
            b_ref,
            native_sigma,
            r1_trap,
            trap_old,
        )
        R += (F / gamma) * dt
        R += sqrt_2Tdt_over_gamma * rng.standard_normal(R.shape)

        dtrap = trap_new - trap_old
        trap_mid = 0.5 * (trap_old + trap_new)
        r_end_mid = 0.5 * (r_end_old + R[:, -1])

        dW = -k_pull * np.einsum(
            "ij,j->i",
            r_end_mid - trap_mid[None, :],
            dtrap,
        )
        cumulative_work += dW

        if t % store_stride == 0:
            mean_power_t[store_idx] = dW.mean() / dt
            mean_work_t[store_idx] = cumulative_work.mean()
            trap_store[store_idx] = trap_new
            store_idx += 1

        if PROGRESS_EVERY > 0 and t % PROGRESS_EVERY == 0:
            report_progress(label, t, n_steps_total - 1, start)

    return R, mean_power_t, mean_work_t, trap_store, cumulative_work


# ================================================================
# Main
# ================================================================
def main():
    overall_start = time.perf_counter()

    print(f"Task ID          = {TASK_ID}", flush=True)
    print(f"N realizations   = {N}", flush=True)
    print(f"RNG seed         = {SEED}", flush=True)
    print(f"Output directory = {OUTPUT_DIR}", flush=True)

    print("Building relaxed model...", flush=True)
    r_guess = make_initial_hairpin_guess()
    r_relaxed = relax_guess_to_folded_shape(r_guess)

    bond_rest, b_ref, native_sigma, r1_trap, _ = (
        build_equilibrium_model_from_relaxed_shape(r_relaxed)
    )

    print("Building trap protocol...", flush=True)
    trap_fwd, trap_rev = build_trap_protocol()

    n_steps_total = len(trap_fwd)
    n_store = (n_steps_total - 1) // store_stride + 1
    time_store = np.arange(n_store, dtype=np.float64) * dt_store

    print(f"n_steps_total    = {n_steps_total}", flush=True)
    print(f"store_stride     = {store_stride}", flush=True)
    print(f"n_store          = {n_store}", flush=True)

    R0 = np.repeat(r_relaxed[None, :, :], N, axis=0)

    print("Thermalizing...", flush=True)
    R0 = thermalize_batch(
        R0,
        bond_rest,
        b_ref,
        native_sigma,
        r1_trap,
        trap_fwd[0],
        rng,
    )

    print("Running forward protocol...", flush=True)
    Rf, mean_power_f, mean_work_f, trap_fwd_store, final_work_f = (
        simulate_protocol_work_batch(
            R0,
            trap_fwd,
            bond_rest,
            b_ref,
            native_sigma,
            r1_trap,
            rng,
            "Forward",
        )
    )

    print("Running reverse protocol...", flush=True)
    _, mean_power_r, mean_work_r, trap_rev_store, final_work_r = (
        simulate_protocol_work_batch(
            Rf,
            trap_rev,
            bond_rest,
            b_ref,
            native_sigma,
            r1_trap,
            rng,
            "Reverse",
        )
    )

    mean_W_f = final_work_f.mean()
    std_W_f = final_work_f.std(ddof=1) if N > 1 else 0.0
    sem_W_f = std_W_f / np.sqrt(N)

    mean_W_r = final_work_r.mean()
    std_W_r = final_work_r.std(ddof=1) if N > 1 else 0.0
    sem_W_r = std_W_r / np.sqrt(N)

    np.save(OUTPUT_DIR / "time_store.npy", time_store)
    np.save(OUTPUT_DIR / "mean_power_f.npy", mean_power_f)
    np.save(OUTPUT_DIR / "mean_power_r.npy", mean_power_r)
    np.save(OUTPUT_DIR / "mean_work_f.npy", mean_work_f)
    np.save(OUTPUT_DIR / "mean_work_r.npy", mean_work_r)
    np.save(OUTPUT_DIR / "final_work_f.npy", final_work_f)
    np.save(OUTPUT_DIR / "final_work_r.npy", final_work_r)
    np.save(OUTPUT_DIR / "trap_fwd_store.npy", trap_fwd_store)
    np.save(OUTPUT_DIR / "trap_rev_store.npy", trap_rev_store)

    elapsed = time.perf_counter() - overall_start

    np.savez(
        OUTPUT_DIR / "batch_summary.npz",
        task_id=TASK_ID,
        seed=SEED,
        N=N,
        dt=dt,
        store_stride=store_stride,
        dt_store=dt_store,
        mean_W_f=mean_W_f,
        std_W_f=std_W_f,
        sem_W_f=sem_W_f,
        mean_W_r=mean_W_r,
        std_W_r=std_W_r,
        sem_W_r=sem_W_r,
        elapsed_seconds=elapsed,
    )

    print("\nTrap work results:", flush=True)
    print(f"  <W_f(final)> = {mean_W_f:.8f} +/- {sem_W_f:.8f} SEM", flush=True)
    print(f"  <W_r(final)> = {mean_W_r:.8f} +/- {sem_W_r:.8f} SEM", flush=True)
    print(f"Elapsed time   = {elapsed / 3600.0:.3f} h", flush=True)
    print(f"Saved outputs in {OUTPUT_DIR}", flush=True)


main()


## 3. Hairpin trajectory snapshots

Visualizes one forward trajectory at $t/\tau=0,0.5,1$, together with a faint ensemble background, native-contact status, and the x-coordinates retained in the coarse-grained representation. This cell uses the trajectory file generated above and saves the figure in the notebook output directory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path

%matplotlib inline

# ------------------------------------------------
# Styling
# ------------------------------------------------
sns.set_style("white")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 25,
    "legend.fontsize": 20,
    "legend.handlelength": 1.8,
    "legend.frameon": False,
    "mathtext.fontset": "stix",
    "mathtext.default": "it",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 200,
    "savefig.dpi": 600,
})

# ------------------------------------------------
# Colors
# ------------------------------------------------
COLOR_BG = "0.0"
COLOR_BACKBONE = "0.15"
COLOR_BEAD = "#0072B2"
COLOR_NATIVE = "#009E73"
COLOR_BROKEN = "#D55E00"
COLOR_AXIS = "black"

# Yellow x-coordinate markers with red edges
COLOR_XMARK = "#FFD600"
COLOR_MARK_EDGE = "red"

# ------------------------------------------------
# Load trajectory generated above
# ------------------------------------------------
traj_fwd = np.load(
    TRAJ_DIR / "traj_fwd.npy",
    mmap_mode="r",
)

n_traj, n_time, n_beads, dim = traj_fwd.shape

if dim != 3:
    raise ValueError(
        f"Expected 3D coordinates, but trajectory dimension is {dim}."
    )

# ------------------------------------------------
# Parameters
# ------------------------------------------------
traj_idx = 6

if not 0 <= traj_idx < n_traj:
    raise IndexError(
        f"traj_idx={traj_idx} is outside the available range "
        f"0 to {n_traj - 1}."
    )

native_pairs_1based = [
    (4, 7),
    (3, 8),
    (2, 9),
]

native_pairs = [
    (i - 1, j - 1)
    for i, j in native_pairs_1based
]

break_factor = 1.5
bead_size = 80

# ------------------------------------------------
# Marker settings
# ------------------------------------------------
XMARK_SIZE = 28
SHOW_PROJECTION_LINES = True

# ------------------------------------------------
# Separate x-limits for the three panels
#
# Order:
#   panel 1: t/tau = 0
#   panel 2: t/tau = 0.5
#   panel 3: t/tau = 1
# ------------------------------------------------
XLIMS = [
    (-1.0, 4.5),
    (-1.0, 7.0),
    (-1.0, 13.0),
]

# Fixed y- and z-limits
YLIM = (-2.0, 2.0)
ZLIM = (-2.0, 4.0)

# Axis-label offsets
X_LABEL_OFFSET = 0.35
Y_LABEL_OFFSET = 0.35
Z_LABEL_OFFSET = 0.35

# ------------------------------------------------
# Three time slices
# ------------------------------------------------
time_idxs = np.array([
    0,
    int(round(0.5 * (n_time - 1))),
    n_time - 1,
])

time_labels = [
    r"$t/\tau = 0$",
    r"$t/\tau = 0.5$",
    r"$t/\tau = 1$",
]

if len(XLIMS) != len(time_idxs):
    raise ValueError(
        "XLIMS must contain one (xmin, xmax) tuple for each time slice."
    )

# ------------------------------------------------
# Background trajectories
# ------------------------------------------------
rng = np.random.default_rng(42)

all_idxs = np.arange(n_traj)
bg_idxs = all_idxs[all_idxs != traj_idx]

n_bg = min(500, len(bg_idxs))

if n_bg > 0:
    bg_trajs = rng.choice(
        bg_idxs,
        size=n_bg,
        replace=False,
    )
else:
    bg_trajs = np.array([], dtype=int)

# ------------------------------------------------
# Native reference distances
# ------------------------------------------------
r0 = np.asarray(
    traj_fwd[traj_idx, 0]
)

native_r0 = np.array([
    np.linalg.norm(r0[i] - r0[j])
    for i, j in native_pairs
])

native_break_dist = break_factor * native_r0

# ------------------------------------------------
# Beads whose x-coordinates will be marked
# ------------------------------------------------
lj_bead_idxs = sorted({
    bead_idx
    for pair in native_pairs
    for bead_idx in pair
})

marked_bead_idxs = sorted({
    0,
    n_beads - 1,
    *lj_bead_idxs,
})

# ------------------------------------------------
# Helpers
# ------------------------------------------------
def set_axis_limits(ax, xlim, ylim, zlim):
    """
    Apply panel-specific x-limits and fixed y- and z-limits.
    """
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]
    z_range = zlim[1] - zlim[0]

    if x_range <= 0 or y_range <= 0 or z_range <= 0:
        raise ValueError("All axis limits must have positive ranges.")

    ax.set_box_aspect((
        x_range,
        y_range,
        z_range,
    ))


def draw_centered_axes(
    ax,
    xlim,
    ylim,
    zlim,
    axis_color="black",
    linewidth=1.4,
):
    """
    Draw custom x, y, and z axes through the origin.
    """
    # x-axis
    ax.plot(
        [xlim[0], xlim[1]],
        [0, 0],
        [0, 0],
        color=axis_color,
        linewidth=linewidth,
        zorder=1,
    )

    # y-axis
    ax.plot(
        [0, 0],
        [ylim[0], ylim[1]],
        [0, 0],
        color=axis_color,
        linewidth=linewidth,
        zorder=1,
    )

    # z-axis
    ax.plot(
        [0, 0],
        [0, 0],
        [zlim[0], zlim[1]],
        color=axis_color,
        linewidth=linewidth,
        zorder=1,
    )

    # Axis labels
    ax.text(
        xlim[1] + X_LABEL_OFFSET,
        0,
        0,
        r"$x$",
        fontsize=19,
        color=axis_color,
        ha="center",
        va="center",
    )

    ax.text(
        0,
        ylim[1] + Y_LABEL_OFFSET,
        0,
        r"$y$",
        fontsize=19,
        color=axis_color,
        ha="center",
        va="center",
    )

    ax.text(
        0,
        0,
        zlim[1] + Z_LABEL_OFFSET,
        r"$z$",
        fontsize=19,
        color=axis_color,
        ha="center",
        va="center",
    )

    # Origin marker
    ax.scatter(
        [0],
        [0],
        [0],
        s=12,
        color=axis_color,
        edgecolors="none",
        depthshade=False,
        zorder=15,
    )

    # Origin label
    ax.text(
        0,
        0,
        0,
        r"$0$",
        fontsize=11,
        color=axis_color,
        ha="right",
        va="top",
        zorder=16,
    )


def draw_axis_ticks(
    ax,
    xlim,
    ylim,
    zlim,
    n_x_ticks=3,
    n_y_ticks=1,
    n_z_ticks=2,
    tick_length=0.08,
    fontsize=15,
):
    """
    Draw custom tick marks and numerical labels.
    """
    x_ticks = np.linspace(
        xlim[0],
        xlim[1],
        n_x_ticks,
    )

    y_ticks = np.linspace(
        ylim[0],
        ylim[1],
        n_y_ticks,
    )

    z_ticks = np.linspace(
        zlim[0],
        zlim[1],
        n_z_ticks,
    )

    # x-axis ticks
    for value in x_ticks:
        if np.isclose(value, 0):
            continue

        ax.plot(
            [value, value],
            [-tick_length, tick_length],
            [0, 0],
            color=COLOR_AXIS,
            linewidth=0.8,
            zorder=2,
        )

        ax.text(
            value,
            -2.5 * tick_length,
            0,
            f"{value:g}",
            fontsize=fontsize,
            color=COLOR_AXIS,
            ha="center",
            va="top",
        )

    # y-axis ticks
    for value in y_ticks:
        if np.isclose(value, 0):
            continue

        ax.plot(
            [-tick_length, tick_length],
            [value, value],
            [0, 0],
            color=COLOR_AXIS,
            linewidth=0.8,
            zorder=2,
        )

        ax.text(
            -2.5 * tick_length,
            value,
            0,
            f"{value:g}",
            fontsize=fontsize,
            color=COLOR_AXIS,
            ha="right",
            va="center",
        )

    # z-axis ticks
    for value in z_ticks:
        if np.isclose(value, 0):
            continue

        ax.plot(
            [-tick_length, tick_length],
            [0, 0],
            [value, value],
            color=COLOR_AXIS,
            linewidth=0.8,
            zorder=2,
        )

        ax.text(
            -2.5 * tick_length,
            0,
            value,
            f"{value:g}",
            fontsize=fontsize,
            color=COLOR_AXIS,
            ha="right",
            va="center",
        )


def mark_x_coordinates(
    ax,
    xyz,
    bead_indices,
    marker_size=18,
    show_projection_lines=False,
):
    """
    Mark selected bead x-coordinates at positions (x_i, 0, 0).
    """
    bead_indices = np.asarray(
        bead_indices,
        dtype=int,
    )

    x_markers = xyz[bead_indices, 0]
    zeros = np.zeros_like(x_markers)

    # Yellow filled points with red edges
    ax.scatter(
        x_markers,
        zeros,
        zeros,
        s=marker_size,
        marker="o",
        facecolors=COLOR_XMARK,
        edgecolors=COLOR_MARK_EDGE,
        linewidths=0.8,
        depthshade=False,
        clip_on=False,
        zorder=20,
    )

    if show_projection_lines:
        for bead_idx in bead_indices:
            ax.plot(
                [
                    xyz[bead_idx, 0],
                    xyz[bead_idx, 0],
                ],
                [
                    xyz[bead_idx, 1],
                    0,
                ],
                [
                    xyz[bead_idx, 2],
                    0,
                ],
                color=COLOR_XMARK,
                linestyle=":",
                linewidth=1.5,
                alpha=0.45,
                zorder=2,
            )


# ------------------------------------------------
# Figure
# ------------------------------------------------
fig = plt.figure(
    figsize=(11, 5.5)
)

axes = [
    fig.add_subplot(
        1,
        3,
        panel_idx + 1,
        projection="3d",
    )
    for panel_idx in range(3)
]

for ax, t, tlabel, panel_xlim in zip(
    axes,
    time_idxs,
    time_labels,
    XLIMS,
):
    xyz = np.asarray(
        traj_fwd[traj_idx, t]
    )

    if n_bg > 0:
        bg_cloud = np.asarray(
            traj_fwd[bg_trajs, t]
        )
    else:
        bg_cloud = np.empty(
            (0, n_beads, 3)
        )

    # Hide Matplotlib's default 3D frame
    ax.set_axis_off()

    # Apply this panel's x-limits
    set_axis_limits(
        ax,
        panel_xlim,
        YLIM,
        ZLIM,
    )

    # Background ensemble
    for xyz_bg in bg_cloud:
        ax.plot(
            xyz_bg[:, 0],
            xyz_bg[:, 1],
            xyz_bg[:, 2],
            color=COLOR_BG,
            alpha=0.015,
            linewidth=0.6,
            zorder=0,
        )

    # Coordinate axes through the origin
    draw_centered_axes(
        ax,
        panel_xlim,
        YLIM,
        ZLIM,
        axis_color=COLOR_AXIS,
        linewidth=1,
    )

    # Custom axis ticks
    draw_axis_ticks(
        ax,
        panel_xlim,
        YLIM,
        ZLIM,
        n_x_ticks=0,
        n_y_ticks=0,
        n_z_ticks=0,
        tick_length=0.08,
        fontsize=14,
    )

    # Backbone
    ax.plot(
        xyz[:, 0],
        xyz[:, 1],
        xyz[:, 2],
        color=COLOR_BACKBONE,
        linewidth=2.4,
        alpha=0.95,
        zorder=5,
    )

    # Native LJ contacts
    for pair_id, (a, b) in enumerate(
        native_pairs
    ):
        rij = np.linalg.norm(
            xyz[a] - xyz[b]
        )

        intact = (
            rij < native_break_dist[pair_id]
        )

        ax.plot(
            [
                xyz[a, 0],
                xyz[b, 0],
            ],
            [
                xyz[a, 1],
                xyz[b, 1],
            ],
            [
                xyz[a, 2],
                xyz[b, 2],
            ],
            color=(
                COLOR_NATIVE
                if intact
                else COLOR_BROKEN
            ),
            linestyle=(
                "--"
                if intact
                else "--"
            ),
            linewidth=(
                3.0
                if intact
                else 1.5
            ),
            alpha=(
                0.95
                if intact
                else 0.70
            ),
            zorder=7,
        )

    # Beads
    ax.scatter(
        xyz[:, 0],
        xyz[:, 1],
        xyz[:, 2],
        s=bead_size,
        color=COLOR_BEAD,
        edgecolors="#2979FF",
        linewidths=.8,
        alpha=0.95,
        depthshade=True,
        zorder=10,
    )

    # Mark selected bead x-coordinates
    mark_x_coordinates(
        ax=ax,
        xyz=xyz,
        bead_indices=marked_bead_idxs,
        marker_size=XMARK_SIZE,
        show_projection_lines=SHOW_PROJECTION_LINES,
    )

    # Time label
    ax.text2D(
        0.7,
        0.77,
        tlabel,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=19,
    )

    # Same camera orientation in every panel
    ax.view_init(
        elev=20,
        azim=-65,
    )

# ------------------------------------------------
# Common legend
# ------------------------------------------------
legend_handles = [
    Line2D(
        [0],
        [0],
        color=COLOR_BACKBONE,
        linewidth=2.4,
        label="Backbone",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_NATIVE,
        linewidth=3.0,
        linestyle="--",
        label="Intact native contact",
    ),
    Line2D(
        [0],
        [0],
        color=COLOR_BROKEN,
        linewidth=2.0,
        linestyle="--",
        label="Broken native contact",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        markersize=10,
        markerfacecolor=COLOR_XMARK,
        markeredgecolor=COLOR_MARK_EDGE,
        markeredgewidth=0.8,
        linestyle="none",
        label=r"Coarse-grained coordinates",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.25),
    ncol=4,
    fontsize=14,
    frameon=False,
    columnspacing=1.5,
    handlelength=2.0,
)

fig.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.98,
    bottom=0.18,
    wspace=-0.05,
)

# ------------------------------------------------
# Save
# ------------------------------------------------
output_path = ROOT / "hairpin_3d_snapshots_separate_xlimits.pdf"

plt.savefig(
    output_path,
    bbox_inches="tight",
    transparent=True,
    pad_inches=0.00,
)

plt.show()
print(f"Saved snapshot figure to: {output_path}")


## 4. Full-coordinate EquiNET inference

Loads the 1000 forward/reverse trajectories from `TRAJ_DIR`, constructs a reproducible disjoint split of 500 training and 500 testing trajectories, trains the forward and reverse full-coordinate EGNNs, and evaluates entropy production only on the independent held-out test set.


In [ ]:
import gc
import os
import random
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from torch import nn
from tqdm import trange

# ================================================================
# NOTEBOOK CONFIGURATION
# ================================================================
data_dir = str(TRAJ_DIR)
output_dir = str(FULL_INFERENCE_DIR)
N_data_requested = N_TRAIN
seed = INFERENCE_SEED

if N_data_requested <= 0:
    raise ValueError(f"N_DATA must be positive, got {N_data_requested}")

os.makedirs(output_dir, exist_ok=True)

print(f"Loading trajectory data from: {data_dir}")
print(f"Saving ML outputs to:        {output_dir}")
print(f"Requested train/test size:   {N_data_requested} each")
print(f"Split seed:                  {seed}")

# ================================================================
# LOAD DATA AS MEMMAP AND CREATE DISJOINT TRAIN/TEST SETS
# ================================================================
traj_fwd_all = np.load(
    os.path.join(data_dir, "traj_fwd.npy"), mmap_mode="r"
)
traj_rev_all = np.load(
    os.path.join(data_dir, "traj_rev.npy"), mmap_mode="r"
)

print("Full traj_fwd shape:", traj_fwd_all.shape)
print("Full traj_rev shape:", traj_rev_all.shape)

if traj_fwd_all.shape != traj_rev_all.shape:
    raise ValueError(
        f"Forward/reverse trajectory shapes do not match: "
        f"{traj_fwd_all.shape} vs {traj_rev_all.shape}"
    )

if traj_fwd_all.ndim != 4:
    raise ValueError(
        "Expected trajectory shape (N, n_store, n_beads, 3). "
        f"Got {traj_fwd_all.shape}"
    )

N_all = traj_fwd_all.shape[0]
required = 2 * N_data_requested
if required > N_all:
    raise ValueError(
        f"Requested N_train=N_test={N_data_requested}, which requires "
        f"{required} trajectories, but only N_all={N_all} are available."
    )

# Reproducible disjoint split. The same indices are used for the
# corresponding forward and reverse trajectory arrays.
rng = np.random.default_rng(seed)
selected_indices = rng.choice(N_all, size=required, replace=False)
train_indices = selected_indices[:N_data_requested]
test_indices = selected_indices[N_data_requested:]

# Materialize only the selected subsets. Advanced indexing creates arrays
# containing the requested trajectories rather than loading the full dataset.
traj_fwd_train = np.asarray(traj_fwd_all[train_indices])
traj_rev_train = np.asarray(traj_rev_all[train_indices])
traj_fwd_test = np.asarray(traj_fwd_all[test_indices])
traj_rev_test = np.asarray(traj_rev_all[test_indices])

N_train = traj_fwd_train.shape[0]
N_eval = traj_fwd_test.shape[0]
_, n_store, n_beads, coord_dim = traj_fwd_train.shape
L = n_store - 1

dt = 1e-4
store_stride = 500
dt_inf = store_stride * dt
t_max = (L - 1) * dt_inf

print("Training traj_fwd shape:", traj_fwd_train.shape)
print("Test traj_fwd shape:    ", traj_fwd_test.shape)
print("Training traj_rev shape:", traj_rev_train.shape)
print("Test traj_rev shape:    ", traj_rev_test.shape)
print(
    f"dt={dt}, dt_inf={dt_inf}, L={L}, "
    f"N_train={N_train}, N_test={N_eval}, "
    f"n_beads={n_beads}, coord_dim={coord_dim}, t_max={t_max:.6f}"
)

# Save the split immediately so every result can be reproduced.
np.save(os.path.join(output_dir, "train_indices.npy"), train_indices)
np.save(os.path.join(output_dir, "test_indices.npy"), test_indices)
np.save(os.path.join(output_dir, "split_seed.npy"), np.array([seed], dtype=np.int64))

# ================================================================
# DEVICE
# ================================================================
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CACHE_DATA_ON_GPU = True

# ================================================================
# ML PARAMETERS
# ================================================================
epochs = 10000
lr = 1e-2
eps_var = 1e-6

K = 30
dim_h = 32
num_blocks = 2

traj_batch = min(5000, N_train)
time_batch = min(10, L)

eval_traj_chunk = min(1000, N_eval)
warmup_epochs = 0

# ================================================================
# MODEL
# ================================================================
def build_edges(n_beads):
    edges = []
    native_pairs = [(3, 6), (2, 7), (1, 8)]

    for i in range(n_beads - 1):
        edges.append((i, i + 1))
        edges.append((i + 1, i))

    for i in range(n_beads - 2):
        edges.append((i, i + 2))
        edges.append((i + 2, i))

    for i, j in native_pairs:
        if i < n_beads and j < n_beads:
            edges.append((i, j))
            edges.append((j, i))

    edges = sorted(set(edges))

    senders = torch.tensor([e[0] for e in edges], dtype=torch.long)
    receivers = torch.tensor([e[1] for e in edges], dtype=torch.long)
    return senders, receivers


class TimeEmbedding(nn.Module):
    def __init__(self, K, t_max):
        super().__init__()
        mu_init = torch.linspace(0.0, t_max, K)
        self.mu = nn.Parameter(mu_init)

        init_sig = max(t_max / max(K, 1), 1e-4)
        self.log_sig = nn.Parameter(
            torch.full((K,), np.log(init_sig), dtype=torch.float32)
        )

    def forward(self, t):
        diff = t.unsqueeze(-1) - self.mu
        sig = self.log_sig.exp().clamp(min=1e-4)
        return torch.exp(-0.5 * (diff / sig) ** 2)


class EGNNLayer(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * h + 1, h),
            nn.SiLU(),
            nn.Linear(h, h),
            nn.SiLU(),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(2 * h, h),
            nn.SiLU(),
            nn.Linear(h, h),
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(h, h),
            nn.SiLU(),
            nn.Linear(h, 1),
        )

    def forward(self, node_h, x, senders, receivers):
        xi = x[:, :, senders, :]
        xj = x[:, :, receivers, :]
        hi = node_h[:, :, senders, :]
        hj = node_h[:, :, receivers, :]

        rij = xi - xj
        dist2 = (rij ** 2).sum(dim=-1, keepdim=True)

        edge_input = torch.cat([hi, hj, dist2], dim=-1)
        mij = self.edge_mlp(edge_input)

        coord_weight = self.coord_mlp(mij)
        dx = rij * coord_weight

        coord_update = torch.zeros_like(x)
        coord_update.index_add_(2, receivers, dx)
        x = x + coord_update / max(len(senders), 1)

        msg = torch.zeros_like(node_h)
        msg.index_add_(2, receivers, mij)
        node_h = node_h + self.node_mlp(torch.cat([node_h, msg], dim=-1))

        return node_h, x


class EGNNGaussianNet(nn.Module):
    def __init__(self, n_beads, coord_dim, K, h, blocks, t_max):
        super().__init__()
        self.n_beads = n_beads
        self.coord_dim = coord_dim
        self.K = K

        self.time_emb = TimeEmbedding(K, t_max)
        self.bead_emb = nn.Embedding(n_beads, h)
        self.time_lift = nn.Linear(K, h)
        self.layers = nn.ModuleList([EGNNLayer(h) for _ in range(blocks)])
        self.out = nn.Sequential(
            nn.Linear(h, h),
            nn.SiLU(),
            nn.Linear(h, coord_dim),
        )

        senders, receivers = build_edges(n_beads)
        self.register_buffer("senders", senders)
        self.register_buffer("receivers", receivers)

    @property
    def mu(self):
        return self.time_emb.mu

    @property
    def log_sig(self):
        return self.time_emb.log_sig

    def phi(self, t):
        return self.time_emb(t)

    def forward(self, x, t):
        B, M, V, C = x.shape
        bead_ids = torch.arange(V, device=x.device)

        bead_h = self.bead_emb(bead_ids)
        bead_h = bead_h.view(1, 1, V, -1).expand(B, M, V, -1)

        time_h = self.time_lift(self.phi(t))
        time_h = time_h.unsqueeze(2).expand(B, M, V, -1)

        node_h = bead_h + time_h
        x_work = x

        for layer in self.layers:
            node_h, x_work = layer(node_h, x_work, self.senders, self.receivers)

        return self.out(node_h)


def make_network():
    net = EGNNGaussianNet(
        n_beads=n_beads,
        coord_dim=coord_dim,
        K=K,
        h=dim_h,
        blocks=num_blocks,
        t_max=t_max,
    ).to(device)

    opt = optim.Adam(net.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=epochs,
        eta_min=1e-5,
    )
    return net, opt, sched


net_fwd, opt_fwd, sched_fwd = make_network()
net_rev, opt_rev, sched_rev = make_network()

# ================================================================
# HELPERS
# ================================================================
def traj_to_mid_diff(traj, label):
    mid = 0.5 * (traj[:, :-1, :, :] + traj[:, 1:, :, :])
    diff = traj[:, 1:, :, :] - traj[:, :-1, :, :]

    mid_t = torch.from_numpy(mid.astype(np.float32))
    diff_t = torch.from_numpy(diff.astype(np.float32))

    print(
        f"  {label}: mid/diff shape = {tuple(mid_t.shape)} "
        f"({mid_t.numel() * mid_t.element_size() / 1e6:.0f} MB each)"
    )

    if CACHE_DATA_ON_GPU and device.type == "cuda":
        try:
            mid_t = mid_t.to(device, non_blocking=True)
            diff_t = diff_t.to(device, non_blocking=True)
            print(f"  {label}: cached mid/diff on GPU")
        except RuntimeError as exc:
            print(f"  {label}: GPU cache failed, keeping data on CPU")
            print(f"  Reason: {exc}")
            mid_t = mid_t.cpu()
            diff_t = diff_t.cpu()
            torch.cuda.empty_cache()

    return mid_t, diff_t


def make_time_axis(L, reverse=False):
    t = torch.arange(L, device=device, dtype=torch.float32) * dt_inf
    return t.flip(0) if reverse else t


time_fwd = make_time_axis(L, reverse=False)
time_rev = make_time_axis(L, reverse=True)


def stratified_time_sample(n):
    return random.sample(range(L), min(n, L))


def compute_jj(net, midyn_t, diffyn_t, time_axis, traj_idx, time_idx):
    if not torch.is_tensor(traj_idx):
        traj_idx = torch.tensor(traj_idx, dtype=torch.long)

    if midyn_t.device.type == "cuda":
        traj_idx = traj_idx.to(midyn_t.device, non_blocking=True)
        dm = midyn_t[traj_idx][:, time_idx, :, :]
        dx = diffyn_t[traj_idx][:, time_idx, :, :]
    else:
        dm = midyn_t[traj_idx][:, time_idx, :, :].to(device, non_blocking=True)
        dx = diffyn_t[traj_idx][:, time_idx, :, :].to(device, non_blocking=True)

    t = time_axis[time_idx].unsqueeze(0).expand(dm.shape[0], -1)
    out = net(dm, t)
    return (out * dx).sum(dim=(2, 3))

# ================================================================
# TRAINING
# ================================================================
def train(net, opt, sched, midyn_t, diffyn_t, time_axis, label):
    net.train()
    ep_loss = []

    for epoch in trange(epochs, desc=f"Training {label}"):
        if midyn_t.device.type == "cuda":
            traj_idx = torch.randperm(N_train, device=midyn_t.device)[:traj_batch]
        else:
            traj_idx = torch.randperm(N_train)[:traj_batch]

        time_idx = stratified_time_sample(time_batch)
        jj = compute_jj(net, midyn_t, diffyn_t, time_axis, traj_idx, time_idx)

        mean_jj = jj.mean(dim=0)
        var_jj = jj.var(dim=0, unbiased=True)

        if epoch < warmup_epochs:
            loss = -mean_jj.sum()
        else:
            loss = (-2.0 * mean_jj**2 / (dt_inf * (var_jj + eps_var))).sum()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
        opt.step()
        sched.step()

        ep_loss.append(float(loss.detach().cpu()))

    return ep_loss

# ================================================================
# EVALUATION OVER THE INDEPENDENT HELD-OUT TEST SET
# ================================================================
def eval_sigma_from_raw(net, traj_data, time_axis, label):
    """Evaluate sigma(t) over the held-out trajectories only."""
    net.eval()

    n_eval_local = traj_data.shape[0]
    sigma = np.zeros(L, dtype=np.float32)

    with torch.inference_mode():
        for t_idx in trange(
            L,
            desc=f"Eval {label} over all {n_eval_local} trajectories",
        ):
            welf_n = 0
            welf_mean = torch.zeros(1, device=device, dtype=torch.float32)
            welf_M2 = torch.zeros(1, device=device, dtype=torch.float32)

            for tr_start in range(0, n_eval_local, eval_traj_chunk):
                tr_end = min(tr_start + eval_traj_chunk, n_eval_local)

                x0_np = traj_data[tr_start:tr_end, t_idx, :, :].astype(np.float32)
                x1_np = traj_data[tr_start:tr_end, t_idx + 1, :, :].astype(np.float32)

                mid_np = 0.5 * (x0_np + x1_np)
                diff_np = x1_np - x0_np

                mid_t = torch.from_numpy(mid_np).to(device, non_blocking=True)
                diff_t = torch.from_numpy(diff_np).to(device, non_blocking=True)

                mid_t = mid_t[:, None, :, :]
                diff_t = diff_t[:, None, :, :]

                t = time_axis[t_idx].view(1, 1).expand(mid_t.shape[0], 1)
                out = net(mid_t, t)
                jj = (out * diff_t).sum(dim=(2, 3))

                chunk_n = jj.shape[0]
                chunk_mean = jj.mean(dim=0)

                if chunk_n > 1:
                    chunk_M2 = jj.var(dim=0, unbiased=True) * (chunk_n - 1)
                else:
                    chunk_M2 = torch.zeros_like(chunk_mean)

                new_n = welf_n + chunk_n
                delta = chunk_mean - welf_mean

                welf_mean = welf_mean + delta * chunk_n / new_n
                welf_M2 = (
                    welf_M2
                    + chunk_M2
                    + delta**2 * welf_n * chunk_n / new_n
                )
                welf_n = new_n

                del x0_np, x1_np, mid_np, diff_np
                del mid_t, diff_t, t, out, jj

            var_jj = welf_M2 / max(welf_n - 1, 1)
            sigma[t_idx] = (
                2.0 * welf_mean**2 / (dt_inf * (var_jj + eps_var))
            ).detach().cpu().item()

    return sigma

# ================================================================
# FORWARD
# ================================================================
print("\n=== Reconstructing forward mid/diff ===")
mid_fwd, diff_fwd = traj_to_mid_diff(traj_fwd_train, "forward train")

print("\n=== Training forward memory-safe 3D EGNN ===")
loss_fwd = train(net_fwd, opt_fwd, sched_fwd, mid_fwd, diff_fwd, time_fwd, "forward")

del mid_fwd, diff_fwd
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n=== Evaluating forward sigma(t) ===")
sigma_fwd = eval_sigma_from_raw(net_fwd, traj_fwd_test, time_fwd, "forward test")

if device.type == "cuda":
    torch.cuda.empty_cache()

# ================================================================
# REVERSE
# ================================================================
print("\n=== Reconstructing reverse mid/diff ===")
mid_rev, diff_rev = traj_to_mid_diff(traj_rev_train, "reverse train")

print("\n=== Training reverse memory-safe 3D EGNN ===")
loss_rev = train(net_rev, opt_rev, sched_rev, mid_rev, diff_rev, time_rev, "reverse")

del mid_rev, diff_rev
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n=== Evaluating reverse sigma(t) ===")
sigma_rev = eval_sigma_from_raw(net_rev, traj_rev_test, time_rev, "reverse test")

if device.type == "cuda":
    torch.cuda.empty_cache()

# ================================================================
# PLOT
# ================================================================
time_arr = np.arange(L, dtype=np.float32) * dt_inf
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(time_arr, sigma_fwd, lw=0.8, color="steelblue", label=r"$\sigma_\mathrm{fwd}(t)$")
axes[0, 0].set_title(f"Forward EPR (N_test={N_eval})")
axes[0, 0].set_xlabel("t [s]")
axes[0, 0].set_ylabel(r"$\sigma(t)$")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid()

axes[0, 1].plot(time_arr, sigma_rev[::-1], lw=0.8, color="tomato", label=r"$\sigma_\mathrm{rev}(t)$")
axes[0, 1].set_title("Reverse EPR - mapped to forward time")
axes[0, 1].set_xlabel("t [s]")
axes[0, 1].set_ylabel(r"$\sigma(t)$")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid()

axes[1, 0].plot(time_arr, sigma_fwd, lw=0.8, color="steelblue", label="forward")
axes[1, 0].plot(time_arr, sigma_rev[::-1], lw=0.8, color="tomato", label="reverse")
axes[1, 0].set_title("EPR - forward and reverse")
axes[1, 0].set_xlabel("t [s]")
axes[1, 0].set_ylabel(r"$\sigma(t)$")
axes[1, 0].legend()
axes[1, 0].grid()

axes[1, 1].semilogy(loss_fwd, color="steelblue", lw=0.7, label="forward")
axes[1, 1].semilogy(loss_rev, color="tomato", lw=0.7, label="reverse")
axes[1, 1].axvline(warmup_epochs, color="gray", lw=0.8, linestyle="--", label="warm-up end")
axes[1, 1].set_title("Training Loss")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].legend(fontsize=7)
axes[1, 1].grid()

t_plot = torch.linspace(0, t_max, 500, device=device)
with torch.inference_mode():
    phi_fwd = net_fwd.phi(t_plot).detach().cpu().numpy()
    phi_rev = net_rev.phi(t_plot).detach().cpu().numpy()

t_np = t_plot.detach().cpu().numpy()

axes[0, 2].set_title("Learned Gaussian bases - forward")
for k_idx in range(K):
    axes[0, 2].plot(t_np, phi_fwd[:, k_idx], lw=0.6, alpha=0.6)
axes[0, 2].set_xlabel("t [s]")
axes[0, 2].set_ylabel(r"$\phi_k(t)$")
axes[0, 2].grid()

axes[1, 2].set_title("Learned Gaussian bases - reverse")
for k_idx in range(K):
    axes[1, 2].plot(t_np, phi_rev[:, k_idx], lw=0.6, alpha=0.6)
axes[1, 2].set_xlabel("t [s]")
axes[1, 2].set_ylabel(r"$\phi_k(t)$")
axes[1, 2].grid()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "epr_fwd_rev_3d_egnn.png"), dpi=150)
plt.close()

# ================================================================
# SAVE RESULTS
# ================================================================
np.save(os.path.join(output_dir, "sigma_fwd.npy"), sigma_fwd)
np.save(os.path.join(output_dir, "sigma_rev.npy"), sigma_rev)
np.save(os.path.join(output_dir, "loss_fwd.npy"), np.array(loss_fwd, dtype=np.float32))
np.save(os.path.join(output_dir, "loss_rev.npy"), np.array(loss_rev, dtype=np.float32))
np.save(os.path.join(output_dir, "time_arr.npy"), time_arr)
np.save(os.path.join(output_dir, "dt_inf.npy"), np.array([dt_inf], dtype=np.float32))
np.save(os.path.join(output_dir, "N_train.npy"), np.array([N_train], dtype=np.int64))
np.save(os.path.join(output_dir, "N_test.npy"), np.array([N_eval], dtype=np.int64))

torch.save(net_fwd.state_dict(), os.path.join(output_dir, "net_fwd_3d_egnn.pt"))
torch.save(net_rev.state_dict(), os.path.join(output_dir, "net_rev_3d_egnn.pt"))

mu_fwd = net_fwd.mu.detach().cpu().numpy()
sig_fwd = net_fwd.log_sig.exp().detach().cpu().numpy()
mu_rev = net_rev.mu.detach().cpu().numpy()
sig_rev = net_rev.log_sig.exp().detach().cpu().numpy()

np.savez(
    os.path.join(output_dir, "gaussian_params_3d_egnn.npz"),
    mu_fwd=mu_fwd,
    sig_fwd=sig_fwd,
    mu_rev=mu_rev,
    sig_rev=sig_rev,
    N_train=np.array([N_train], dtype=np.int64),
    N_test=np.array([N_eval], dtype=np.int64),
    split_seed=np.array([seed], dtype=np.int64),
)

# ================================================================
# SUMMARY
# ================================================================
total_fwd = float(np.sum(sigma_fwd) * dt_inf)
total_rev = float(np.sum(sigma_rev) * dt_inf)

print(f"\nIndependent split: N_train={N_train}, N_test={N_eval}, seed={seed}")
print(f"Total EP forward   : {total_fwd:.4f} kB")
print(f"Total EP reverse   : {total_rev:.4f} kB")
print(f"Sum (should be >=0): {total_fwd + total_rev:.4f} kB")

print(f"\nForward Gaussian centres (mu_k):\n  {np.round(mu_fwd, 4)}")
print(f"Forward Gaussian widths  (sig_k):\n  {np.round(sig_fwd, 4)}")
print(f"\nResults saved to: {output_dir}")
print("Done.")


## 5. Coarse-grained / partial-observation EquiNET inference

Uses only the x coordinates of the selected beads (native-contact beads plus the terminal/anchor beads, as in the supplied CG code). The notebook uses the same 500/500 disjoint held-out split as requested.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import random
import os
import sys
import gc
from torch import nn
from tqdm import trange

# ================================================================
# NOTEBOOK CONFIGURATION + HELD-OUT SPLIT
# ================================================================
data_dir = str(TRAJ_DIR)
output_dir = str(CG_INFERENCE_DIR)
N_data_requested = N_TRAIN
seed = INFERENCE_SEED

os.makedirs(output_dir, exist_ok=True)

print(f"Loading trajectory data from: {data_dir}")
print(f"Saving ML outputs to:        {output_dir}")
print(f"Requested train/test size:   {N_data_requested} each")
print(f"Split seed:                  {seed}")

# ================================================================
# LOAD FULL DATA AS MEMMAP
# ================================================================
traj_fwd_all = np.load(os.path.join(data_dir, "traj_fwd.npy"), mmap_mode="r")
traj_rev_all = np.load(os.path.join(data_dir, "traj_rev.npy"), mmap_mode="r")

print("traj_fwd_all shape:", traj_fwd_all.shape)
print("traj_rev_all shape:", traj_rev_all.shape)

if traj_fwd_all.shape != traj_rev_all.shape:
    raise ValueError(
        f"Forward/reverse trajectory shapes do not match: "
        f"{traj_fwd_all.shape} vs {traj_rev_all.shape}"
    )

if traj_fwd_all.ndim != 4:
    raise ValueError(
        "Expected trajectory shape (N, n_store, n_beads, 3). "
        f"Got {traj_fwd_all.shape}"
    )

N_all, n_store, n_beads_full, coord_dim_full = traj_fwd_all.shape
L = n_store - 1

required = 2 * N_data_requested
if required > N_all:
    raise ValueError(
        f"Requested N_train=N_test={N_data_requested}, which requires "
        f"{required} trajectories, but only N_all={N_all} are available."
    )

rng = np.random.default_rng(seed)
selected_indices = rng.choice(N_all, size=required, replace=False)
train_indices = selected_indices[:N_data_requested]
test_indices = selected_indices[N_data_requested:]

# ================================================================
# USE ONLY X COORDINATES OF BEADS IN NATIVE CONTACTS + LAST BEAD
# ================================================================
native_pairs = [(3, 6), (2, 7), (1, 8)]

beads_with_native_contacts = sorted(
    set(i for pair in native_pairs for i in pair)
)

selected_bead_ids = sorted(
    set(beads_with_native_contacts + [n_beads_full - 1])
)

selected_bead_ids_np = np.array(selected_bead_ids, dtype=np.int64)

n_beads = len(selected_bead_ids)
coord_dim = 1

print("Native pairs:", native_pairs)
print("Using only x coordinates of original beads:", selected_bead_ids)
print(f"Reduced input shape per frame: n_beads={n_beads}, coord_dim={coord_dim}")

traj_fwd_train = np.asarray(traj_fwd_all[train_indices][:, :, selected_bead_ids_np, 0:1])
traj_rev_train = np.asarray(traj_rev_all[train_indices][:, :, selected_bead_ids_np, 0:1])
traj_fwd_test = np.asarray(traj_fwd_all[test_indices])
traj_rev_test = np.asarray(traj_rev_all[test_indices])

N_train = traj_fwd_train.shape[0]
N_eval = traj_fwd_test.shape[0]

dt = 1e-4
store_stride = 500
dt_inf = store_stride * dt
t_max = (L - 1) * dt_inf

print(
    f"dt={dt}, dt_inf={dt_inf}, L={L}, "
    f"N_train={N_train}, N_eval={N_eval}, "
    f"n_beads={n_beads}, coord_dim={coord_dim}, t_max={t_max:.6f}"
)

np.save(os.path.join(output_dir, "train_indices.npy"), train_indices)
np.save(os.path.join(output_dir, "test_indices.npy"), test_indices)
np.save(os.path.join(output_dir, "split_seed.npy"), np.array([seed], dtype=np.int64))

# ================================================================
# DEVICE
# ================================================================
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CACHE_DATA_ON_GPU = True

# ================================================================
# ML PARAMETERS
# ================================================================
epochs = 10000
lr = 1e-2
eps_var = 1e-6

K = 30
dim_h = 32
num_blocks = 2

traj_batch = min(5000, N_train)
time_batch = min(10, L)

eval_traj_chunk = 1000
warmup_epochs = 0

# ================================================================
# MODEL
# ================================================================
def build_edges(selected_bead_ids):
    selected_bead_ids = list(selected_bead_ids)
    selected_set = set(selected_bead_ids)
    old_to_new = {old: new for new, old in enumerate(selected_bead_ids)}

    edges = []
    candidate_pairs = []

    # Bonded neighbors in original bead indexing
    for i in range(n_beads_full - 1):
        candidate_pairs.append((i, i + 1))

    # Next-nearest neighbors in original bead indexing
    for i in range(n_beads_full - 2):
        candidate_pairs.append((i, i + 2))

    # Native contacts in original bead indexing
    candidate_pairs.extend(native_pairs)

    for i, j in candidate_pairs:
        if i in selected_set and j in selected_set:
            ii = old_to_new[i]
            jj = old_to_new[j]
            edges.append((ii, jj))
            edges.append((jj, ii))

    # Fallback if selected beads have no edges left
    if len(edges) == 0 and len(selected_bead_ids) > 1:
        for i in range(len(selected_bead_ids) - 1):
            edges.append((i, i + 1))
            edges.append((i + 1, i))

    edges = sorted(set(edges))

    senders = torch.tensor([e[0] for e in edges], dtype=torch.long)
    receivers = torch.tensor([e[1] for e in edges], dtype=torch.long)

    print("Reduced graph edges:", edges)

    return senders, receivers


class TimeEmbedding(nn.Module):
    def __init__(self, K, t_max):
        super().__init__()

        mu_init = torch.linspace(0.0, t_max, K)
        self.mu = nn.Parameter(mu_init)

        init_sig = max(t_max / max(K, 1), 1e-4)
        self.log_sig = nn.Parameter(
            torch.full((K,), np.log(init_sig), dtype=torch.float32)
        )

    def forward(self, t):
        diff = t.unsqueeze(-1) - self.mu
        sig = self.log_sig.exp().clamp(min=1e-4)
        return torch.exp(-0.5 * (diff / sig) ** 2)


class EGNNLayer(nn.Module):
    def __init__(self, h):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * h + 1, h),
            nn.SiLU(),
            nn.Linear(h, h),
            nn.SiLU(),
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(2 * h, h),
            nn.SiLU(),
            nn.Linear(h, h),
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(h, h),
            nn.SiLU(),
            nn.Linear(h, 1),
        )

    def forward(self, node_h, x, senders, receivers):
        xi = x[:, :, senders, :]
        xj = x[:, :, receivers, :]

        hi = node_h[:, :, senders, :]
        hj = node_h[:, :, receivers, :]

        rij = xi - xj
        dist2 = (rij ** 2).sum(dim=-1, keepdim=True)

        edge_input = torch.cat([hi, hj, dist2], dim=-1)
        mij = self.edge_mlp(edge_input)

        coord_weight = self.coord_mlp(mij)
        dx = rij * coord_weight

        coord_update = torch.zeros_like(x)
        coord_update.index_add_(2, receivers, dx)

        x = x + coord_update / max(len(senders), 1)

        msg = torch.zeros_like(node_h)
        msg.index_add_(2, receivers, mij)

        node_h = node_h + self.node_mlp(torch.cat([node_h, msg], dim=-1))

        return node_h, x


class EGNNGaussianNet(nn.Module):
    def __init__(self, n_beads, coord_dim, K, h, blocks, t_max):
        super().__init__()

        self.n_beads = n_beads
        self.coord_dim = coord_dim
        self.K = K

        self.time_emb = TimeEmbedding(K, t_max)
        self.bead_emb = nn.Embedding(n_beads, h)
        self.time_lift = nn.Linear(K, h)

        self.layers = nn.ModuleList([EGNNLayer(h) for _ in range(blocks)])

        self.out = nn.Sequential(
            nn.Linear(h, h),
            nn.SiLU(),
            nn.Linear(h, coord_dim),
        )

        senders, receivers = build_edges(selected_bead_ids)
        self.register_buffer("senders", senders)
        self.register_buffer("receivers", receivers)

    @property
    def mu(self):
        return self.time_emb.mu

    @property
    def log_sig(self):
        return self.time_emb.log_sig

    def phi(self, t):
        return self.time_emb(t)

    def forward(self, x, t):
        B, M, V, C = x.shape

        bead_ids = torch.arange(V, device=x.device)

        bead_h = self.bead_emb(bead_ids)
        bead_h = bead_h.view(1, 1, V, -1).expand(B, M, V, -1)

        time_h = self.time_lift(self.phi(t))
        time_h = time_h.unsqueeze(2).expand(B, M, V, -1)

        node_h = bead_h + time_h

        x_work = x
        for layer in self.layers:
            node_h, x_work = layer(node_h, x_work, self.senders, self.receivers)

        drift = self.out(node_h)

        return drift


def make_network():
    net = EGNNGaussianNet(
        n_beads=n_beads,
        coord_dim=coord_dim,
        K=K,
        h=dim_h,
        blocks=num_blocks,
        t_max=t_max,
    ).to(device)

    opt = optim.Adam(net.parameters(), lr=lr)

    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=epochs,
        eta_min=1e-5,
    )

    return net, opt, sched


net_fwd, opt_fwd, sched_fwd = make_network()
net_rev, opt_rev, sched_rev = make_network()

# ================================================================
# HELPERS
# ================================================================
def traj_to_mid_diff(traj, label):
    mid = 0.5 * (traj[:, :-1, :, :] + traj[:, 1:, :, :])
    diff = traj[:, 1:, :, :] - traj[:, :-1, :, :]

    mid_t = torch.from_numpy(mid.astype(np.float32))
    diff_t = torch.from_numpy(diff.astype(np.float32))

    print(
        f"  {label}: mid/diff shape = {tuple(mid_t.shape)} "
        f"({mid_t.numel() * mid_t.element_size() / 1e6:.0f} MB each)"
    )

    if CACHE_DATA_ON_GPU and device.type == "cuda":
        try:
            mid_t = mid_t.to(device, non_blocking=True)
            diff_t = diff_t.to(device, non_blocking=True)
            print(f"  {label}: cached mid/diff on GPU")
        except RuntimeError as e:
            print(f"  {label}: GPU cache failed, keeping data on CPU")
            print(f"  Reason: {e}")
            mid_t = mid_t.cpu()
            diff_t = diff_t.cpu()
            torch.cuda.empty_cache()

    return mid_t, diff_t


def make_time_axis(L, reverse=False):
    t = torch.arange(L, device=device, dtype=torch.float32) * dt_inf
    return t.flip(0) if reverse else t


time_fwd = make_time_axis(L, reverse=False)
time_rev = make_time_axis(L, reverse=True)


def stratified_time_sample(n):
    return random.sample(range(L), min(n, L))


def compute_jj(net, midyn_t, diffyn_t, time_axis, traj_idx, time_idx):
    if not torch.is_tensor(traj_idx):
        traj_idx = torch.tensor(traj_idx, dtype=torch.long)

    if midyn_t.device.type == "cuda":
        traj_idx = traj_idx.to(midyn_t.device, non_blocking=True)
        dm = midyn_t[traj_idx][:, time_idx, :, :]
        dx = diffyn_t[traj_idx][:, time_idx, :, :]
    else:
        dm = midyn_t[traj_idx][:, time_idx, :, :].to(device, non_blocking=True)
        dx = diffyn_t[traj_idx][:, time_idx, :, :].to(device, non_blocking=True)

    t = time_axis[time_idx].unsqueeze(0).expand(dm.shape[0], -1)

    out = net(dm, t)

    return (out * dx).sum(dim=(2, 3))


# ================================================================
# TRAINING
# ================================================================
def train(net, opt, sched, midyn_t, diffyn_t, time_axis, label):
    net.train()
    ep_loss = []

    for epoch in trange(epochs, desc=f"Training {label}"):
        if midyn_t.device.type == "cuda":
            traj_idx = torch.randperm(N_train, device=midyn_t.device)[:traj_batch]
        else:
            traj_idx = torch.randperm(N_train)[:traj_batch]

        time_idx = stratified_time_sample(time_batch)

        jj = compute_jj(net, midyn_t, diffyn_t, time_axis, traj_idx, time_idx)

        mean_jj = jj.mean(dim=0)
        var_jj = jj.var(dim=0, unbiased=True)

        if epoch < warmup_epochs:
            loss = -mean_jj.sum()
        else:
            loss = (-2.0 * mean_jj**2 / (dt_inf * (var_jj + eps_var))).sum()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
        opt.step()
        sched.step()

        ep_loss.append(float(loss.detach().cpu()))

    return ep_loss


# ================================================================
# HELD-OUT STREAMING EVALUATION
# ================================================================
def eval_sigma_from_raw(net, traj_all, time_axis, label):
    """
    Evaluate sigma(t) over held-out trajectories using only:
      - x coordinates of beads in native contacts
      - x coordinate of the last bead
    """
    net.eval()

    N_eval_local = traj_all.shape[0]
    sigma = np.zeros(L, dtype=np.float32)

    with torch.inference_mode():
        for t_idx in trange(L, desc=f"Eval {label} over all {N_eval_local} trajectories"):

            welf_n = 0
            welf_mean = torch.zeros(1, device=device, dtype=torch.float32)
            welf_M2 = torch.zeros(1, device=device, dtype=torch.float32)

            for tr_start in range(0, N_eval_local, eval_traj_chunk):
                tr_end = min(tr_start + eval_traj_chunk, N_eval_local)

                x0_np = traj_all[
                    tr_start:tr_end,
                    t_idx,
                    selected_bead_ids_np,
                    0:1,
                ].astype(np.float32)

                x1_np = traj_all[
                    tr_start:tr_end,
                    t_idx + 1,
                    selected_bead_ids_np,
                    0:1,
                ].astype(np.float32)

                mid_np = 0.5 * (x0_np + x1_np)
                diff_np = x1_np - x0_np

                mid_t = torch.from_numpy(mid_np).to(device, non_blocking=True)
                diff_t = torch.from_numpy(diff_np).to(device, non_blocking=True)

                mid_t = mid_t[:, None, :, :]
                diff_t = diff_t[:, None, :, :]

                t = time_axis[t_idx].view(1, 1).expand(mid_t.shape[0], 1)

                out = net(mid_t, t)
                jj = (out * diff_t).sum(dim=(2, 3))

                chunk_n = jj.shape[0]
                chunk_mean = jj.mean(dim=0)

                if chunk_n > 1:
                    chunk_M2 = jj.var(dim=0, unbiased=True) * (chunk_n - 1)
                else:
                    chunk_M2 = torch.zeros_like(chunk_mean)

                new_n = welf_n + chunk_n
                delta = chunk_mean - welf_mean

                welf_mean = welf_mean + delta * chunk_n / new_n
                welf_M2 = (
                    welf_M2
                    + chunk_M2
                    + delta**2 * welf_n * chunk_n / new_n
                )

                welf_n = new_n

                del x0_np, x1_np, mid_np, diff_np
                del mid_t, diff_t, t, out, jj

            var_jj = welf_M2 / max(welf_n - 1, 1)

            sigma[t_idx] = (
                2.0 * welf_mean**2 / (dt_inf * (var_jj + eps_var))
            ).detach().cpu().item()

    return sigma


# ================================================================
# FORWARD
# ================================================================
print("\n=== Reconstructing forward mid/diff for training subset ===")
mid_fwd, diff_fwd = traj_to_mid_diff(traj_fwd_train, "forward")

print("\n=== Training forward memory-safe x-only native-contact EGNN ===")
loss_fwd = train(net_fwd, opt_fwd, sched_fwd, mid_fwd, diff_fwd, time_fwd, "forward")

del mid_fwd, diff_fwd
gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n=== Evaluating forward sigma(t) over held-out trajectories ===")
sigma_fwd = eval_sigma_from_raw(net_fwd, traj_fwd_test, time_fwd, "forward")

if device.type == "cuda":
    torch.cuda.empty_cache()

# ================================================================
# REVERSE
# ================================================================
print("\n=== Reconstructing reverse mid/diff for training subset ===")
mid_rev, diff_rev = traj_to_mid_diff(traj_rev_train, "reverse")

print("\n=== Training reverse memory-safe x-only native-contact EGNN ===")
loss_rev = train(net_rev, opt_rev, sched_rev, mid_rev, diff_rev, time_rev, "reverse")

del mid_rev, diff_rev
gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()

print("\n=== Evaluating reverse sigma(t) over held-out trajectories ===")
sigma_rev = eval_sigma_from_raw(net_rev, traj_rev_test, time_rev, "reverse")

if device.type == "cuda":
    torch.cuda.empty_cache()

# ================================================================
# PLOT
# ================================================================
time_arr = np.arange(L, dtype=np.float32) * dt_inf

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(time_arr, sigma_fwd, lw=0.8, color="steelblue", label=r"$\sigma_\mathrm{fwd}(t)$")
axes[0, 0].set_title("Forward EPR")
axes[0, 0].set_xlabel("t [s]")
axes[0, 0].set_ylabel(r"$\sigma(t)$")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid()

axes[0, 1].plot(time_arr, sigma_rev[::-1], lw=0.8, color="tomato", label=r"$\sigma_\mathrm{rev}(t)$")
axes[0, 1].set_title("Reverse EPR — mapped to forward time")
axes[0, 1].set_xlabel("t [s]")
axes[0, 1].set_ylabel(r"$\sigma(t)$")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid()

axes[1, 0].plot(time_arr, sigma_fwd, lw=0.8, color="steelblue", label="forward")
axes[1, 0].plot(time_arr, sigma_rev[::-1], lw=0.8, color="tomato", label="reverse")
axes[1, 0].set_title("EPR — forward and reverse")
axes[1, 0].set_xlabel("t [s]")
axes[1, 0].set_ylabel(r"$\sigma(t)$")
axes[1, 0].legend()
axes[1, 0].grid()

axes[1, 1].semilogy(loss_fwd, color="steelblue", lw=0.7, label="forward")
axes[1, 1].semilogy(loss_rev, color="tomato", lw=0.7, label="reverse")
axes[1, 1].axvline(warmup_epochs, color="gray", lw=0.8, linestyle="--", label="warm-up end")
axes[1, 1].set_title("Training Loss")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].legend(fontsize=7)
axes[1, 1].grid()

t_plot = torch.linspace(0, t_max, 500, device=device)

with torch.inference_mode():
    phi_fwd = net_fwd.phi(t_plot).detach().cpu().numpy()
    phi_rev = net_rev.phi(t_plot).detach().cpu().numpy()

t_np = t_plot.detach().cpu().numpy()

axes[0, 2].set_title("Learned Gaussian bases — forward")
for k_idx in range(K):
    axes[0, 2].plot(t_np, phi_fwd[:, k_idx], lw=0.6, alpha=0.6)
axes[0, 2].set_xlabel("t [s]")
axes[0, 2].set_ylabel(r"$\phi_k(t)$")
axes[0, 2].grid()

axes[1, 2].set_title("Learned Gaussian bases — reverse")
for k_idx in range(K):
    axes[1, 2].plot(t_np, phi_rev[:, k_idx], lw=0.6, alpha=0.6)
axes[1, 2].set_xlabel("t [s]")
axes[1, 2].set_ylabel(r"$\phi_k(t)$")
axes[1, 2].grid()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "epr_fwd_rev_x_native_egnn.png"), dpi=150)
plt.close()

# ================================================================
# SAVE RESULTS
# ================================================================
np.save(os.path.join(output_dir, "sigma_fwd.npy"), sigma_fwd)
np.save(os.path.join(output_dir, "sigma_rev.npy"), sigma_rev)
np.save(os.path.join(output_dir, "loss_fwd.npy"), np.array(loss_fwd, dtype=np.float32))
np.save(os.path.join(output_dir, "loss_rev.npy"), np.array(loss_rev, dtype=np.float32))
np.save(os.path.join(output_dir, "time_arr.npy"), time_arr)
np.save(os.path.join(output_dir, "dt_inf.npy"), np.array([dt_inf], dtype=np.float32))
np.save(os.path.join(output_dir, "selected_bead_ids.npy"), selected_bead_ids_np)
np.save(os.path.join(output_dir, "N_train.npy"), np.array([N_train], dtype=np.int64))
np.save(os.path.join(output_dir, "N_test.npy"), np.array([N_eval], dtype=np.int64))

torch.save(net_fwd.state_dict(), os.path.join(output_dir, "net_fwd_x_native_egnn.pt"))
torch.save(net_rev.state_dict(), os.path.join(output_dir, "net_rev_x_native_egnn.pt"))

mu_fwd = net_fwd.mu.detach().cpu().numpy()
sig_fwd = net_fwd.log_sig.exp().detach().cpu().numpy()
mu_rev = net_rev.mu.detach().cpu().numpy()
sig_rev = net_rev.log_sig.exp().detach().cpu().numpy()

np.savez(
    os.path.join(output_dir, "gaussian_params_x_native_egnn.npz"),
    mu_fwd=mu_fwd,
    sig_fwd=sig_fwd,
    mu_rev=mu_rev,
    sig_rev=sig_rev,
    selected_bead_ids=selected_bead_ids_np,
    N_train=np.array([N_train], dtype=np.int64),
    N_test=np.array([N_eval], dtype=np.int64),
    split_seed=np.array([seed], dtype=np.int64),
)

# ================================================================
# SUMMARY
# ================================================================
total_fwd = float(np.sum(sigma_fwd) * dt_inf)
total_rev = float(np.sum(sigma_rev) * dt_inf)

print(f"\nIndependent split: N_train={N_train}, N_test={N_eval}, seed={seed}")
print(f"Total EP forward   : {total_fwd:.4f} kB")
print(f"Total EP reverse   : {total_rev:.4f} kB")
print(f"Sum (should be >=0): {total_fwd + total_rev:.4f} kB")

print(f"\nSelected original bead ids:\n  {selected_bead_ids}")
print(f"\nForward Gaussian centres (mu_k):\n  {np.round(mu_fwd, 4)}")
print(f"Forward Gaussian widths  (sig_k):\n  {np.round(sig_fwd, 4)}")

print(f"\nResults saved to: {output_dir}")
print("Done.")


## 6. Final cumulative-estimate plot

This cell reproduces the requested visual style using the outputs generated above. The cumulative full and coarse-grained entropy-production curves are obtained directly from the inferred `sigma_fwd` arrays. The work curve is loaded from the work-distribution calculation.

Because this notebook performs one inference run rather than an ensemble of repeated inference runs, it does not have the across-run standard-deviation bands used in the original plotting snippet. The three mean curves are therefore shown without artificial uncertainty bands.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator

# -----------------------------------------------------
# Style
# -----------------------------------------------------

plt.rcParams.update({
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.linewidth": 1.5,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 24,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "mathtext.fontset": "cm",
})

# -----------------------------------------------------
# Paths
# -----------------------------------------------------

# ROOT is defined near the beginning of the Colab notebook.
#
# For example:
# ROOT = Path("/content/equinet_hairpin_notebook_run")

full_dir = ROOT / "inference_full"
work_dir = ROOT / "work_distribution"

# -----------------------------------------------------
# Time resolutions
# -----------------------------------------------------

dt = 1e-4

# EquiNET trajectory storage:
inference_store_stride = 500
dt_inf = inference_store_stride * dt

# Work calculation storage:
work_store_stride = 50
dt_work = work_store_stride * dt

# Ratio required to place work on the EquiNET time grid
work_downsample = inference_store_stride // work_store_stride

print(f"EquiNET dt = {dt_inf}")
print(f"Work dt    = {dt_work}")
print(f"Work downsampling factor = {work_downsample}")

# -----------------------------------------------------
# Load full entropy-production estimate
# -----------------------------------------------------

sigma_file = full_dir / "sigma_fwd.npy"

if not sigma_file.exists():
    raise FileNotFoundError(
        f"Forward entropy-production file not found: {sigma_file}"
    )

sigma_fwd = np.asarray(
    np.load(sigma_file)
).squeeze()

# Time associated with sigma(t)
time_full = (
    np.arange(len(sigma_fwd), dtype=np.float64)
    * dt_inf
)

# Cumulative entropy production:
#
#     Delta S_tot(t) / k_B
#         = integral_0^t sigma(t') dt'
#
fwd_full = np.cumsum(sigma_fwd) * dt_inf

# -----------------------------------------------------
# Load cumulative work
# -----------------------------------------------------

work_file = work_dir / "mean_work_f.npy"

if not work_file.exists():
    raise FileNotFoundError(
        f"Forward-work file not found: {work_file}"
    )

work_fwd = np.asarray(
    np.load(work_file)
).squeeze()

# Work is stored every 500 steps whereas the trajectory used
# for EquiNET is stored every 5000 steps.
#
# Downsample by 10 so the two curves have the same resolution.
work_fwd_plot = work_fwd[::work_downsample]

# -----------------------------------------------------
# Match array lengths
# -----------------------------------------------------

n_common = min(
    len(time_full),
    len(fwd_full),
    len(work_fwd_plot),
)

time_plot = time_full[:n_common]
fwd_full_plot = fwd_full[:n_common]
work_fwd_plot = work_fwd_plot[:n_common]

print("Number of plotted points:", n_common)
print(
    "Final cumulative entropy production:",
    f"{fwd_full_plot[-1]:.4f} k_B",
)
print(
    "Final work:",
    f"{work_fwd_plot[-1]:.4f} k_B T",
)

# -----------------------------------------------------
# Plot
# -----------------------------------------------------

color = "tab:blue"

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

# -----------------------------------------------------
# Full cumulative entropy production
# -----------------------------------------------------

ax.plot(
    time_plot,
    fwd_full_plot,
    color=color,
    linewidth=2.7,
    linestyle="-",
    alpha=1.0,
)

# -----------------------------------------------------
# Work
# -----------------------------------------------------

ax.plot(
    time_plot,
    work_fwd_plot,
    color=color,
    linewidth=2.7,
    linestyle=":",
)

# -----------------------------------------------------
# Formatting
# -----------------------------------------------------

ax.set_xlabel(
    r"$t/\tau$"
)

ax.set_ylabel(
    r"Cumulative Estimates ($k_\mathrm{B} T$)",
    fontsize=22,
)

ax.grid(False)

# -----------------------------------------------------
# Minor ticks
# -----------------------------------------------------

ax.xaxis.set_minor_locator(
    AutoMinorLocator(2)
)

ax.yaxis.set_minor_locator(
    AutoMinorLocator(2)
)

ax.minorticks_on()

# Major ticks
ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=6.0,
    width=1.5,
    pad=5,
    bottom=True,
    left=True,
    top=False,
    right=False,
    color="black",
)

# Minor ticks
ax.tick_params(
    axis="both",
    which="minor",
    direction="out",
    length=3.0,
    width=1.0,
    bottom=True,
    left=True,
    top=False,
    right=False,
    color="black",
)

# -----------------------------------------------------
# Spines
# -----------------------------------------------------

ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

ax.spines["bottom"].set_linewidth(1.5)
ax.spines["left"].set_linewidth(1.5)

# -----------------------------------------------------
# Legend
# -----------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        color=color,
        linewidth=2.7,
        linestyle="-",
        label=(
            r"$\langle \Delta S_{\mathrm{tot}}"
            r"\rangle/k_\mathrm{B}$"
        ),
    ),
    Line2D(
        [0],
        [0],
        color=color,
        linewidth=2.7,
        linestyle=":",
        label=r"$W/k_\mathrm{B}T$",
    ),
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    loc="best",
    fontsize=24,
)

fig.tight_layout()

# -----------------------------------------------------
# Save
# -----------------------------------------------------

output_path = (
    ROOT
    / "eps3_N500_forward_work_dS_tot.pdf"
)

fig.savefig(
    output_path,
    bbox_inches="tight",
    transparent=True,
    pad_inches=0.04,
)

# Optional high-resolution PNG as well
fig.savefig(
    ROOT / "eps3_N500_forward_work_dS_tot.png",
    bbox_inches="tight",
    transparent=True,
    pad_inches=0.04,
    dpi=300,
)

plt.show()
display(fig)
plt.close(fig)
print(f"Saved figure to:\n{output_path}")

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator
from IPython.display import display

# -----------------------------------------------------
# Style
# -----------------------------------------------------
plt.rcParams.update({
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.linewidth": 1.5,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 24,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "mathtext.fontset": "cm",
})

# -----------------------------------------------------
# Load outputs from this notebook
# -----------------------------------------------------
full_sigma = np.asarray(np.load(FULL_INFERENCE_DIR / "sigma_fwd.npy")).squeeze()
cg_sigma = np.asarray(np.load(CG_INFERENCE_DIR / "sigma_fwd.npy")).squeeze()
full_time = np.asarray(np.load(FULL_INFERENCE_DIR / "time_arr.npy")).squeeze()
cg_time = np.asarray(np.load(CG_INFERENCE_DIR / "time_arr.npy")).squeeze()
full_dt_inf = float(np.asarray(np.load(FULL_INFERENCE_DIR / "dt_inf.npy")).squeeze())
cg_dt_inf = float(np.asarray(np.load(CG_INFERENCE_DIR / "dt_inf.npy")).squeeze())

# Cumulative inferred entropy production.
fwd_full = np.cumsum(full_sigma) * full_dt_inf
fwd_cg = np.cumsum(cg_sigma) * cg_dt_inf

# The inference time array labels increments; keep exactly the matching number of points.
time_full = full_time[:len(fwd_full)]
time_cg = cg_time[:len(fwd_cg)]

# -----------------------------------------------------
# Cumulative work
# -----------------------------------------------------
work_file = WORK_DIR / "mean_work_f.npy"
if not work_file.exists():
    raise FileNotFoundError(f"Forward-work file not found: {work_file}")

work_fwd = np.asarray(np.load(work_file)).squeeze()

# Same downsampling used in the supplied plotting code.
work_fwd_plot = work_fwd[::10]
n_work = min(len(time_full), len(work_fwd_plot))
time_work = time_full[:n_work]
work_fwd_plot = work_fwd_plot[:n_work]

# -----------------------------------------------------
# Plot
# -----------------------------------------------------
color = "tab:blue"
fig, ax = plt.subplots(figsize=(9, 5.5))

# Full Delta S_tot -- faint background curve
ax.plot(
    time_full,
    fwd_full,
    color=color,
    linewidth=1.0,
    linestyle="-",
    alpha=0.20,
    zorder=1,
)

# Coarse-grained Delta S_tot^x
ax.plot(
    time_cg,
    fwd_cg,
    color=color,
    linewidth=2.7,
    linestyle="--",
    zorder=3,
)

# Work
ax.plot(
    time_work,
    work_fwd_plot,
    color=color,
    linewidth=2.7,
    linestyle=":",
    zorder=4,
)

# -----------------------------------------------------
# Formatting
# -----------------------------------------------------
ax.set_xlabel(r"$t/\tau$")
ax.set_ylabel(r"Cumulative Estimates ($k_\mathrm{B} T$)", fontsize=22)
ax.grid(False)

ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.minorticks_on()

ax.tick_params(
    axis="both", which="major", direction="out",
    length=6.0, width=1.5, pad=5,
    bottom=True, left=True, top=False, right=False, color="black",
)
ax.tick_params(
    axis="both", which="minor", direction="out",
    length=3.0, width=1.0,
    bottom=True, left=True, top=False, right=False, color="black",
)

ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.spines["bottom"].set_linewidth(1.5)
ax.spines["left"].set_linewidth(1.5)

legend_handles = [
    Line2D(
        [0], [0], color=color, linewidth=2.7, linestyle="--",
        label=r"$\langle \Delta S_{\mathrm{tot}}^{x}\rangle/k_\mathrm{B}$",
    ),
    Line2D(
        [0], [0], color=color, linewidth=2.7, linestyle=":",
        label=r"$W/k_\mathrm{B}T$",
    ),
    Line2D(
        [0], [0], color=color, linewidth=1.5, linestyle="-", alpha=0.35,
        label=r"$\langle \Delta S_{\mathrm{tot}}\rangle/k_\mathrm{B}$",
    ),
]
ax.legend(handles=legend_handles, frameon=False, loc="best", fontsize=24)

fig.tight_layout()

output_pdf = ROOT / "eps3_N500_forward_work_dS_tot_CG.pdf"
output_png = ROOT / "eps3_N500_forward_work_dS_tot_CG.png"
fig.savefig(output_pdf, bbox_inches="tight", transparent=True, pad_inches=0.04)
fig.savefig(output_png, bbox_inches="tight", transparent=True, pad_inches=0.04)

print(f"Saved PDF: {output_pdf}")
print(f"Saved PNG: {output_png}")
display(fig)
plt.close(fig)


In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D

# ------------------------------------------------
# Style
# ------------------------------------------------

sns.set_style("white")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.labelsize": 8,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.2,
    "legend.handlelength": 1.5,
    "legend.frameon": False,
    "mathtext.fontset": "stix",
    "mathtext.default": "it",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 200,
    "savefig.dpi": 600,
})

COLOR_FWD = "#0072B2"
COLOR_REV = "#D55E00"

# ------------------------------------------------
# Paths
# ------------------------------------------------

work_dir = ROOT / "work_distribution"
full_dir = ROOT / "inference_full"
cg_dir = ROOT / "inference_cg"

# ------------------------------------------------
# Load work distributions
# ------------------------------------------------

work_f = np.load(
    work_dir / "final_work_f.npy"
).ravel()

work_r = np.load(
    work_dir / "final_work_r.npy"
).ravel()

minus_work_r = -work_r

# ------------------------------------------------
# Load full and coarse-grained entropy production
# ------------------------------------------------

sigma_fwd_full = np.load(
    full_dir / "sigma_fwd.npy"
).squeeze()

sigma_rev_full = np.load(
    full_dir / "sigma_rev.npy"
).squeeze()

sigma_fwd_cg = np.load(
    cg_dir / "sigma_fwd.npy"
).squeeze()

sigma_rev_cg = np.load(
    cg_dir / "sigma_rev.npy"
).squeeze()

# ------------------------------------------------
# Convert integrated entropy production to Delta F
# ------------------------------------------------

dt = 1e-4
store_stride_inf = 500
dt_inf = store_stride_inf * dt

EP_fwd_full = np.sum(sigma_fwd_full) * dt_inf
EP_rev_full = np.sum(sigma_rev_full) * dt_inf

EP_fwd_cg = np.sum(sigma_fwd_cg) * dt_inf
EP_rev_cg = np.sum(sigma_rev_cg) * dt_inf

Wf = work_f.mean()
Wr = work_r.mean()

deltaF_fwd_full = Wf - EP_fwd_full
deltaF_rev_full = EP_rev_full - Wr

deltaF_fwd_cg = Wf - EP_fwd_cg
deltaF_rev_cg = EP_rev_cg - Wr

print(f"<W>_F = {Wf:.5f}")
print(f"<W>_R = {Wr:.5f}")
print(f"DeltaF full forward = {deltaF_fwd_full:.5f}")
print(f"DeltaF full reverse = {deltaF_rev_full:.5f}")
print(f"DeltaF CG forward   = {deltaF_fwd_cg:.5f}")
print(f"DeltaF CG reverse   = {deltaF_rev_cg:.5f}")

# ------------------------------------------------
# Plot
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(3.3, 2.4)
)

sns.histplot(
    work_f,
    bins=20,
    stat="density",
    color=COLOR_FWD,
    alpha=0.35,
    kde=True,
    line_kws={"lw": 1.6},
    ax=ax,
)

sns.histplot(
    minus_work_r,
    bins=20,
    stat="density",
    color=COLOR_REV,
    alpha=0.35,
    kde=True,
    line_kws={"lw": 1.6},
    ax=ax,
)

# Full estimates
ax.axvline(
    deltaF_fwd_full,
    color=COLOR_FWD,
    linestyle="--",
    linewidth=1.4,
)

ax.axvline(
    deltaF_rev_full,
    color=COLOR_REV,
    linestyle="--",
    linewidth=1.4,
)

# Coarse-grained estimates
ax.axvline(
    deltaF_fwd_cg,
    color=COLOR_FWD,
    linestyle=":",
    linewidth=1.3,
)

ax.axvline(
    deltaF_rev_cg,
    color=COLOR_REV,
    linestyle=":",
    linewidth=1.3,
)

ax.text(
    0.03,
    0.96,
    r"$\epsilon/k_{\rm B}T=3$",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8,
)

ax.set_xlabel(
    r"$W/k_{\rm B}T$"
)

ax.set_ylabel(
    "Density"
)

# Automatic x-range from data
combined = np.concatenate(
    [work_f, minus_work_r]
)

xmin, xmax = np.percentile(
    combined,
    [0.5, 99.5]
)

pad = 0.08 * (xmax - xmin)

ax.set_xlim(
    xmin - pad,
    xmax + pad,
)

ax.grid(False)

ax.xaxis.set_minor_locator(
    AutoMinorLocator(2)
)

ax.yaxis.set_minor_locator(
    AutoMinorLocator(2)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=2,
    width=0.6,
    pad=1.5,
)

ax.tick_params(
    axis="both",
    which="minor",
    direction="out",
    length=1,
    width=0.4,
)

legend_handles = [
    Line2D(
        [0], [0],
        color=COLOR_FWD,
        lw=1.6,
        label=r"$P_{\rm F}(W)$",
    ),
    Line2D(
        [0], [0],
        color=COLOR_REV,
        lw=1.6,
        label=r"$P_{\rm R}(-W)$",
    ),
    Line2D(
        [0], [0],
        color=COLOR_FWD,
        lw=1.3,
        linestyle="--",
        label=r"$\Delta F^{\rm f}$",
    ),
    Line2D(
        [0], [0],
        color=COLOR_REV,
        lw=1.3,
        linestyle="--",
        label=r"$\Delta F^{\rm r}$",
    ),
    Line2D(
        [0], [0],
        color=COLOR_FWD,
        lw=1.2,
        linestyle=":",
        label=r"$\Delta F_x^{\rm f}$",
    ),
    Line2D(
        [0], [0],
        color=COLOR_REV,
        lw=1.2,
        linestyle=":",
        label=r"$\Delta F_x^{\rm r}$",
    ),
]

ax.legend(
    handles=legend_handles,
    loc="best",
    fontsize=5.8,
    frameon=False,
)

fig.tight_layout()

output_path = (
    ROOT
    / "eps3_work_histogram_DeltaF.pdf"
)

fig.savefig(
    output_path,
    bbox_inches="tight",
    transparent=True,
    pad_inches=0.02,
)

plt.show()
display(fig)

print(f"Saved to: {output_path}")

## Output locations

All outputs are written below `equinet_hairpin_notebook_run/` (under `/content/` on Colab unless Google Drive is enabled):

- `trajectory_data/` — forward/reverse stored trajectories and trap protocols;
- `work_distribution/` — work-distribution outputs;
- `inference_full/` — full-coordinate EquiNET outputs;
- `inference_cg/` — coarse-grained EquiNET outputs;
- `eps3_N500_forward_work_dS_tot_CG.pdf` and `.png` — final cumulative-estimate figure.
